# GeoNUSAF Part 2 / Stage 2 — Mask-conditioned satellite image generation (SD1.5 + LoRA + trained control branch)

**Kathmandu Valley land-use painting — 7-channel one-hot condition, 512x512**

Structural twin of `segformer-b0-new.ipynb` and `geonusaf-stage1-maskddpm.ipynb`: same 25 sections,
same config-dataclass pattern, same asserts, same persistence. Only the modelling sections
(9, 11-14, 17-19, 21, 24) differ, so the three notebooks diff cleanly. The dataset, the filename
convention, the colour LUT and the fold-0 split are carried over **unchanged** — deliberately,
because Stage 1 and Stage 2 must land on exactly the same tiles.

## What this stage produces
A generator that paints a realistic satellite image onto a **given** land-use mask. It trains on
REAL masks from the fold-0 train split, so it does not wait on Stage 1; the two are chained only at
sampling time. Output is paired `(image, mask)` PNGs that drop straight into part-1 segmentation
training. The point of the whole exercise is the rare classes — **River (0.57% of pixels),
Road (2.28%) and UnusedLand (IoU ~0.18-0.30)**.

## What is fixed by protocol (do not change per-model)
| Item | Value |
|---|---|
| Dataset | 408 base tiles x 3 flip variants = 1224 pairs, unchangeable |
| Split | fold **0** of the 3-fold sequence-block CV, `BLOCK_BUFFER=2` dropped from train only |
| Leakage rule | the generator trains on **fold-0 train tiles only**; val tiles never enter a gradient |
| Colour -> index | LUT applied at **full 2345x2351 resolution**, then `INTER_NEAREST` |
| Ignore | index 255 on disk, carried as channel 6 of the condition |
| Augmentation | rot90 (k in 0..3) only; hflip/vflip are baked into the 1224 files and are NOT repeated |
| Seed | single seed = 42 |
| Determinism | **off** (see section 3) |
| Runtime | Kaggle free T4 x1, fp16 AMP, gradient checkpointing, 8-bit AdamW, 12 h session cap |
| Effective batch | 16 (`BATCH_SIZE` x `GRAD_ACCUM`; the split is a memory knob, the product is not) |
| Input contract | class-index PNGs, single channel, 512x512, values `{0..5}` + `255` (or `6`) for ignore |

## What is model-specific here
- **Stable Diffusion 1.5, frozen.** `runwayml/stable-diffusion-v1-5` was removed from the Hub in
  August 2024, so this notebook uses the community mirror
  **`stable-diffusion-v1-5/stable-diffusion-v1-5`** — same weights, maintained repo. Set
  `cfg.SD_MODEL_ID` if you prefer a different mirror.
- **LoRA (rank 16) on the UNet attention projections**, lr 1e-4. This is the only thing that teaches
  the model what Kathmandu looks like from above — rooftop texture, terraced fields, river sediment.
  Base weights never move.
- **A control branch warm-started from `lllyasviel/control_v11p_sd15_seg`**, with its conditioning
  input conv replaced so it accepts **7 one-hot channels** instead of 3 RGB channels, zero-initialized,
  fine-tuned at lr 1e-5.
- **The condition format is the fix, and it is the whole point.** The old pipeline fed the project's
  BGR palette to an off-the-shelf seg ControlNet that only understands the ADE20K palette, so the
  mask carried no meaning and was effectively ignored. One-hot also removes the false colour metric
  where Road `(0,128,0)` sits "closer to" Forest `(0,128,128)` than to anything it resembles.
  Section 9 prints that distance matrix rather than asserting it.
- **No MiDaS depth and no external reference image.** Depth is invalid on nadir orthorectified tiles
  and the tile geo-coordinates are unrecoverable, so a real DEM is impossible; the reference image
  transplants a foreign city's layout. Both are gone, not flagged off.

## How this run is judged
Three numbers decide whether the generator is usable, and they are all in section 19:
1. **Layout fidelity** — generate from a held-out real val mask, re-segment the generated image with
   the frozen part-1 SegFormer-B0, and take mIoU against the mask it was conditioned on. Reported
   three ways: matched (the number), **shuffled** (the null — chance level for this class prior),
   and **real images** (the ceiling — B0 only scores 0.2877 on real fold-0 val images, so read the
   ratio, not the absolute).
2. **KID** between generated and real validation images, N stated explicitly. FID is reported too
   but is secondary — at a few hundred real scenes its estimator bias is large and
   generator-dependent.
3. **clDice on Road and River** between the conditioning mask and the re-segmented output, because
   a 9 px road that survives generation as a smear is the failure that mIoU is least sensitive to.

## Session strategy (12 h Kaggle cap)
Full optimizer/scaler/scheduler state is pushed to Hugging Face as `last.pt`; on restart the notebook
resumes from it automatically. `MAX_HOURS` stops the loop before Kaggle kills the session, so a
checkpoint always survives.

## Prerequisites
- **Internet ON** (Settings -> Internet). Needed for pip, the SD/ControlNet weights, W&B and HF.
- Kaggle Secrets: `HF_TOKEN` (write scope) and `WANDB_API_KEY`.
- GPU: T4 x1.


## 1. Environment

`diffusers`, `peft` and `bitsandbytes` are not on the Kaggle image; `ptflops` is not either.
Versions are printed rather than pinned beyond a floor — `ControlNetModel` and
`UNet2DConditionModel.add_adapter` have had a stable API since diffusers 0.25 — but record the
printed versions in the paper's reproducibility section.

`bitsandbytes` is the one import allowed to fail: if it does, section 12 falls back to `torch.AdamW`
and prints that it has done so. The fallback costs about 0.7 GB of VRAM for the control branch's
optimizer state, which still fits, so a broken wheel is an inconvenience rather than a blocker.

### The torchao shim
The Kaggle image ships **torchao 0.10**, and current `peft` versions declare anything below **0.16**
incompatible. That would be harmless except that `peft`'s LoRA injection walks a fixed list of
dispatchers, one of which is `dispatch_torchao`, and that dispatcher calls `is_torchao_available()`
*before* deciding it is not applicable — and `is_torchao_available()` **raises** on an out-of-range
version rather than returning `False`. So `unet.add_adapter(...)` in section 12 dies with
`ImportError: Found an incompatible version of torchao` on a notebook that never asked for torchao
at all.

Nothing here uses torchao: the LoRA targets are ordinary `nn.Linear` attention projections, so the
correct answer is "not available" and the ordinary dispatcher takes over. The shim below reports
exactly that. It is applied at import time and prints which modules it touched, so it is visible in
the log rather than being silent magic.

The alternative fixes both cost more: `pip install -U torchao` pulls a build matched to a newer
torch than the Kaggle image has, and `pip uninstall -y torchao` needs a kernel restart to take
effect on an already-imported module. Neither buys anything, since the target modules are plain
Linear layers either way.


In [ ]:
!pip install -q "diffusers>=0.27.0" "peft>=0.10.0" "bitsandbytes>=0.43.0" ptflops==0.7.3 2>&1 | tail -2

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # set BEFORE cuda initialises

import re, gc, json, time, math, random, warnings, itertools, copy, hashlib, tarfile, shutil, csv
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)   # critical: cv2 threads fight DataLoader workers on Kaggle's 4 vCPUs

import diffusers, transformers, peft

# ---- torchao shim: see the note above ----------------------------------------------------
# peft's LoRA dispatcher list includes dispatch_torchao, which calls is_torchao_available()
# before deciding it does not apply -- and that function RAISES on Kaggle's torchao 0.10
# instead of returning False. Nothing here uses torchao, so report it unavailable and let the
# ordinary nn.Linear dispatcher handle the attention projections.
import importlib
def _disable_torchao_in_peft():
    hit = []
    for name in ("peft.import_utils", "peft.tuners.lora.torchao", "peft.tuners.lora.model",
                 "peft.tuners.tuners_utils", "peft.utils.other"):
        try: mod = importlib.import_module(name)
        except Exception: continue
        if getattr(mod, "is_torchao_available", None) is not None:
            mod.is_torchao_available = lambda: False
            hit.append(name.split(".")[-1])
    return hit

try:
    import torchao
    _tao = getattr(torchao, "__version__", "?")
except Exception:
    _tao = None
_shim = _disable_torchao_in_peft()
print(f"torchao {_tao} | peft torchao-dispatcher shim applied to: {_shim or 'nothing (not needed)'}")

try:
    import bitsandbytes as bnb
    HAS_BNB = True
except Exception as e:
    HAS_BNB = False
    print("bitsandbytes unavailable, will fall back to torch.AdamW:", type(e).__name__, e)

print("torch", torch.__version__, "| diffusers", diffusers.__version__,
      "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| bnb", HAS_BNB)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 2. Configuration

**Stage 2 runs on fold 0 only.** That is not a convenience, it is the leakage rule: the generator
may only ever see fold-0 training tiles, because the augmentation experiment in stage 4 is reported
on fold 0. Training on all 408 tiles and then evaluating on a held-out fold would make every
downstream number meaningless, so `FOLD` and `SPLIT_MODE` are here for parity with the segmentation
notebooks rather than as knobs to sweep.

### The ablation ladder
`ABLATION` is the one field to change between runs. It resolves into four flags — condition format,
LoRA on/off, control branch trained/frozen, canny branch on/off — so that every row of the paper's
table is a config change and not a code edit.

| Row | Condition | LoRA | Control branch | Canny | Needs its own training run |
|---|---|---|---|---|---|
| **A** | project palette RGB | no | off-the-shelf | yes | no — evaluate only |
| **B** | ADE20K palette remap | no | off-the-shelf | yes | no — evaluate only |
| **C** | ADE20K palette remap | yes | off-the-shelf | yes | **yes** |
| **D** | **7-channel one-hot** | yes | **trained** | yes | **yes** — the main method |
| **E** | 7-channel one-hot | yes | trained | **no** | no — re-uses D's weights |

A and B train nothing, so section 18 skips straight past training. E is a **sampling-time** ablation
of D: the canny branch is frozen and pretrained and is never in the training graph, so E loads D's
checkpoint and only changes the sampler. That saves a whole training run and is why `HF_REPO` is
keyed on `WEIGHTS_ABLATION` rather than on `ABLATION`.

The canny condition is derived from the **class boundaries of the mask itself**, not from an
external reference image — that was the part of the old pipeline that transplanted Barcelona's
street grid into Kathmandu, and it is gone.


In [ ]:
@dataclass
class CFG:
    # ---- run selection (fixed for stage 2 -- see the leakage rule) ----
    SPLIT_MODE: str = "block"          # stage 2 is block-only; "random" would break the protocol
    FOLD: int = 0                      # generator trains on fold 0's TRAIN tiles only
    ABLATION: str = "D"                # A | B | C | D | E  -- resolves into the four flags below

    # ---- data ----
    DATA_ROOT: str = "/kaggle/input/datasets/sugam24/satellite-datasets/satellite_dataset"
    CACHE_DIR: str = "/kaggle/working/cache512"
    # If you saved the 512 cache as a Kaggle dataset (the segmentation notebooks build the same
    # one), point this at it to skip the repack entirely.
    PREBUILT_CACHE: Optional[str] = None
    IMG_SIZE: int = 512
    LATENT_SIZE: int = 64              # IMG_SIZE / 8, the SD VAE downsample factor
    N_CLASSES: int = 6
    IGNORE_INDEX: int = 255            # on disk, and in the interface contract
    IGNORE_CODE: int = 6               # condition channel index for ignore
    N_COND_CH: int = 7                 # 6 classes + ignore, one-hot
    PRELOAD_RAM: bool = True
    FREE_IMAGES_AFTER_CACHE: bool = True   # images are only needed to build the latent cache

    # ---- split ----
    N_FOLDS: int = 3
    BLOCK_BUFFER: int = 2              # base tiles dropped from train at each block boundary
    STAGE1_TRAIN_SHA1: Optional[str] = None   # paste Stage 1's printed sha1 to hard-assert agreement

    # ---- condition (set by ABLATION -- do not edit by hand) ----
    COND_MODE: str = "onehot"          # "onehot" | "palette" | "ade20k"
    COND_CONV_INIT: str = "zeros"      # "zeros" | "palette_project"  (onehot only)
    USE_CANNY: bool = True             # sampling-time only; never in the training graph
    COND_SCALE: float = 1.0            # conditioning scale of the seg/one-hot branch
    CANNY_SCALE: float = 0.4           # conditioning scale of the frozen canny branch

    # ---- base checkpoints ----
    # runwayml/stable-diffusion-v1-5 was removed from the Hub in Aug 2024; this is the mirror.
    SD_MODEL_ID: str = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    CONTROLNET_ID: str = "lllyasviel/control_v11p_sd15_seg"
    CANNY_ID: str = "lllyasviel/control_v11p_sd15_canny"

    # ---- LoRA on the UNet attention projections ----
    USE_LORA: bool = True
    LORA_RANK: int = 16                # 16-32; 16 is enough for 810 images
    LORA_ALPHA: int = 16               # alpha == rank -> scaling 1.0
    LORA_DROPOUT: float = 0.0
    LR_LORA: float = 1e-4

    # ---- control branch ----
    TRAIN_CONTROLNET: bool = True
    LR_CONTROL: float = 1e-5

    # ---- text ----
    CAPTION_DROP: float = 0.1          # drop to "" this often, so CFG has an unconditional branch

    # ---- diffusion ----
    DDIM_STEPS: int = 30               # bulk sampling
    GUIDANCE: float = 7.5
    SNR_GAMMA: Optional[float] = 5.0   # min-SNR-gamma loss weighting; None = plain MSE

    # ---- optimization ----
    # Effective batch is BATCH_SIZE x GRAD_ACCUM = 16 and is part of the protocol; the split
    # between the two is a pure memory knob. See section 12.
    BATCH_SIZE: int = 2                # micro-batch actually resident on the GPU
    GRAD_ACCUM: int = 8                # -> effective batch 16
    MAX_STEPS: int = 3000              # OPTIMIZER steps, not epochs. ~60 passes over 810 images
    WARMUP_STEPS: int = 200
    CLIP_NORM: float = 1.0
    WEIGHT_DECAY: float = 1e-2
    NO_DECAY_ON_NORM_BIAS: bool = True
    USE_8BIT_ADAM: bool = True
    AMP: bool = True
    GRAD_CHECKPOINT: bool = True       # on BOTH the UNet and the control branch
    AUG_ROT90: bool = True

    # ---- monitoring ----
    VAL_MONITOR: bool = True           # fixed-noise eps-MSE on val tiles, NO gradient
    VAL_NOISE_SEED: int = 1234
    VAL_EVERY: int = 100               # optimizer steps between val passes
    VAL_BATCHES: int = 24              # cap the val pass so it stays under ~20 s
    SAMPLE_EVERY: int = 500            # optimizer steps between sample grids
    N_SAMPLE_GRID: int = 4
    LOG_EVERY: int = 25

    # ---- evaluation ----
    SEG_REPO: str = "Pranilllllll/segformer-b0-kaggle-ktm-dataset-block-fold0"
    EVAL_N: int = 128                  # generated val scenes used for layout fidelity, KID and FID
    EVAL_BATCH: int = 4
    KID_SUBSET_SIZE: int = 50
    KID_SUBSETS: int = 100
    LAYOUT_MIOU_FLOOR: float = 0.30    # absolute floor, as specified
    LAYOUT_RATIO_FLOOR: float = 0.60   # gen mIoU / real-image mIoU -- the honest reading
    RUN_UNTRAINED_ABLATIONS: bool = False   # section 24: evaluate rows A and B in this session
    ABLATION_EVAL_N: int = 64

    # ---- sampling interface (section 21) ----
    SAMPLE_IN_DIR: Optional[str] = None     # dir of class-index PNGs; None -> fold-0 val masks
    SAMPLE_OUT_DIR: str = "/kaggle/working/fake_pairs"
    N_SAMPLE_OUT: int = 200
    SAMPLE_SEED: int = 777

    # ---- runtime ----
    SEED: int = 42
    NUM_WORKERS: int = 2
    MAX_HOURS: float = 10.5            # stop before Kaggle's 12 h cap

    # ---- persistence ----
    HF_USER: str = "Pranilllllll"           # <-- set to your HF username
    HF_PREFIX: str = "geonusaf-stage2-mask2image"
    PUSH_LAST_EVERY: int = 500         # optimizer steps between full-state pushes (~1.5 GB each)
    PUSH_OPTIM_STATE: bool = True      # False halves last.pt at the cost of Adam moments on resume
    WANDB_PROJECT: str = "geonusaf-baselines"
    OUT_DIR: str = "/kaggle/working/out"
    GEN_DIR: str = "/kaggle/working/gen_eval"

    CLASS_NAMES: tuple = ("Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural")

cfg = CFG()

# ---- resolve the ablation into flags -------------------------------------------------------
ABLATIONS = {
    "A": dict(COND_MODE="palette", USE_LORA=False, TRAIN_CONTROLNET=False, USE_CANNY=True),
    "B": dict(COND_MODE="ade20k",  USE_LORA=False, TRAIN_CONTROLNET=False, USE_CANNY=True),
    "C": dict(COND_MODE="ade20k",  USE_LORA=True,  TRAIN_CONTROLNET=False, USE_CANNY=True),
    "D": dict(COND_MODE="onehot",  USE_LORA=True,  TRAIN_CONTROLNET=True,  USE_CANNY=True),
    "E": dict(COND_MODE="onehot",  USE_LORA=True,  TRAIN_CONTROLNET=True,  USE_CANNY=False),
}
assert cfg.ABLATION in ABLATIONS, cfg.ABLATION
for k, v in ABLATIONS[cfg.ABLATION].items():
    setattr(cfg, k, v)

os.makedirs(cfg.OUT_DIR, exist_ok=True)
os.makedirs(cfg.GEN_DIR, exist_ok=True)
os.makedirs(cfg.SAMPLE_OUT_DIR, exist_ok=True)

COND_CH   = cfg.N_COND_CH if cfg.COND_MODE == "onehot" else 3
CODE_NAMES = list(cfg.CLASS_NAMES) + ["Ignore"]         # 7 condition channels, index == channel
TRAINABLE = bool(cfg.USE_LORA or cfg.TRAIN_CONTROLNET)  # A and B train nothing at all

# E re-uses D's weights: the canny branch is frozen and is a sampling-time knob only.
WEIGHTS_ABLATION = "D" if cfg.ABLATION == "E" else cfg.ABLATION
RUN_NAME = f"{cfg.HF_PREFIX}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}-{WEIGHTS_ABLATION}"
HF_REPO  = f"{cfg.HF_USER}/{RUN_NAME}"
tag = f"{cfg.SPLIT_MODE}_fold{cfg.FOLD}_{cfg.ABLATION}"

print("RUN:", RUN_NAME, "| HF repo:", HF_REPO, "| results tag:", tag)
print(f"ablation {cfg.ABLATION}: cond={cfg.COND_MODE} ({COND_CH} ch) | LoRA {cfg.USE_LORA} "
      f"r{cfg.LORA_RANK} | control branch trained {cfg.TRAIN_CONTROLNET} | canny {cfg.USE_CANNY}")
print(f"base: {cfg.SD_MODEL_ID}  +  {cfg.CONTROLNET_ID}")
print(f"batch: {cfg.BATCH_SIZE} x {cfg.GRAD_ACCUM} accum = {cfg.BATCH_SIZE*cfg.GRAD_ACCUM} effective "
      f"| {cfg.MAX_STEPS} optimizer steps | checkpointing {cfg.GRAD_CHECKPOINT} | trainable {TRAINABLE}")

## 3. Determinism — deliberately OFF here

Same setting as Stage 1, for the same reason and it is not an oversight.
`torch.use_deterministic_algorithms(True)` routes part of this backward through a sort-based kernel
and costs roughly **6x wall-clock** on this workload. A diffusion finetune needs thousands of steps
through a 860 M-parameter UNet where a segmentation run needs a few thousand through a 3.8 M one, so
that factor is the difference between finishing inside a session and not finishing at all.

Everything cheap is still pinned — Python/NumPy/torch RNG, worker seeding, the DataLoader generator,
the fixed noise used by the validation monitor, and the per-batch seed inside the sampler.
`cudnn.benchmark=True` is on because every batch has identical shape. State the caveat in the paper:
reproducible up to cuDNN kernel selection, not bit-exact.


In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True          # fixed shapes -> autotune once, reuse
    torch.use_deterministic_algorithms(False)      # ~6x faster than True on this backward

def worker_init_fn(worker_id):
    s = cfg.SEED + worker_id
    np.random.seed(s); random.seed(s)

set_seed(cfg.SEED)
g = torch.Generator(); g.manual_seed(cfg.SEED)
print("seeded:", cfg.SEED, "| deterministic algorithms: OFF | cudnn.benchmark:",
      torch.backends.cudnn.benchmark)

## 4. Filename parsing and base-tile grouping

Identical to the segmentation notebooks and to Stage 1, and copied rather than rewritten for exactly
that reason. Convention: `image{N}.png` / `image{N}hf.png` / `image{N}vf.png`, masks mirror with the
`mask` prefix. `N` is ascending **export order with gaps**, and the three variants of a tile are
*not* adjacent in a sorted listing — so grouping is regex-based on `N`, never positional.

Stage 2 reads both images and masks. The assertions are the guard rail: if a future dataset version
renames or drops anything, this cell fails loudly instead of silently splitting a tile across folds.


In [ ]:
IMG_DIR  = Path(cfg.DATA_ROOT) / "images"
MASK_DIR = Path(cfg.DATA_ROOT) / "masks"
PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")

records = []          # (base_id:int, variant:str, image_name:str, mask_name:str)
for f in sorted(os.listdir(IMG_DIR)):
    m = PAT.match(f)
    assert m, f"unparsed image filename: {f}"
    base_id, variant = int(m.group(1)), (m.group(2) or "orig")
    mask_name = "mask" + f[len("image"):]
    assert (MASK_DIR / mask_name).exists(), f"missing mask for {f}"
    records.append((base_id, variant, f, mask_name))

groups: Dict[int, List[tuple]] = {}
for r in records:
    groups.setdefault(r[0], []).append(r)

BASE_IDS = sorted(groups)                      # ascending export order = the spatial proxy axis
assert len(records) == 1224, len(records)
assert len(BASE_IDS) == 408, len(BASE_IDS)
assert all(len(v) == 3 for v in groups.values())

gaps = [b - a for a, b in zip(BASE_IDS, BASE_IDS[1:]) if b - a > 1]
print(f"{len(records)} files | {len(BASE_IDS)} base tiles | id range {BASE_IDS[0]}..{BASE_IDS[-1]}")
print(f"{len(gaps)} gaps in export sequence (dropped tiles), total {sum(g-1 for g in gaps)} missing ids")
h, w = cv2.imread(str(MASK_DIR / records[0][3]), cv2.IMREAD_COLOR).shape[:2]
print(f"native tile size: {w}x{h} | GSD after resize to {cfg.IMG_SIZE}: "
      f"{300.0/cfg.IMG_SIZE:.3f} m/px")

## 5. Mask colour diagnostic — run this before anything else

Same guard as the segmentation notebooks and Stage 1, and it matters at least as much here: the
condition is built from whatever the LUT hands the model. If Residential and River were swapped by a
channel-order mistake, the generator would paint rooftops where the water is and every metric in
section 19 would still look plausible, because the segmenter would be scored against the same
swapped mask.

Two things this cell reports:
- the **orientation guard** — Residential must dominate River by a wide margin, else the BGR/RGB
  order is inverted;
- the **unmapped share** — the fraction of pixels whose colour is in neither the 6-colour table nor
  the ignore colour. This is required to read **0.000000%**. Anything else means CVAT polygon edge
  artifacts are silently being folded into ignore, and ignore is a *condition channel* here, so the
  model would learn to paint an artifact class.


In [ ]:
BGR_TO_CLASS = {
    (128, 0, 0): 0,    # Residential Area
    (0, 128, 0): 1,    # Road
    (0, 0, 128): 2,    # River
    (0, 128, 128): 3,  # Forest
    (128, 128, 0): 4,  # Unused Land
    (128, 0, 128): 5,  # Agricultural Area
}
IGNORE_BGR = (0, 0, 0)

# Pack BGR into a single uint32 key so the mapping is one vectorized lookup, not 6 boolean passes.
def pack_bgr(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.uint32)
    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]

def _k(bgr): return int(pack_bgr(np.array(bgr, dtype=np.uint8).reshape(1, 1, 3))[0, 0])

LUT   = np.full(1 << 24, cfg.IGNORE_INDEX, dtype=np.uint8)   # 16 MB; default = ignore
KNOWN = np.zeros(1 << 24, dtype=bool)                        # 16 MB; True only for palette colours
for bgr, idx in BGR_TO_CLASS.items():
    LUT[_k(bgr)] = idx; KNOWN[_k(bgr)] = True
KNOWN[_k(IGNORE_BGR)] = True

def rgbmask_to_index(bgr_img: np.ndarray) -> np.ndarray:
    return LUT[pack_bgr(bgr_img)]

# ---- diagnostic over a random sample of raw full-res masks ----
rng = np.random.RandomState(cfg.SEED)
sample = [records[i] for i in rng.choice(len(records), 40, replace=False)]
colour_hits, class_px, unmapped, total = {}, np.zeros(7, dtype=np.int64), 0, 0

for _, _, _, mname in sample:
    m = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)   # BGR
    keys = pack_bgr(m)
    cols, cnts = np.unique(m.reshape(-1, 3), axis=0, return_counts=True)
    for c, n in zip(cols, cnts):
        colour_hits[tuple(int(x) for x in c)] = colour_hits.get(tuple(int(x) for x in c), 0) + int(n)
    idx = LUT[keys]
    class_px += np.bincount(np.where(idx == cfg.IGNORE_INDEX, 6, idx).ravel(), minlength=7)
    unmapped += int((~KNOWN[keys]).sum()); total += keys.size

known = set(BGR_TO_CLASS)
print("distinct BGR colours found in sample:")
for c, n in sorted(colour_hits.items(), key=lambda kv: -kv[1]):
    tag_c = (f"-> class {BGR_TO_CLASS[c]} ({cfg.CLASS_NAMES[BGR_TO_CLASS[c]]})" if c in known
             else ("-> IGNORE (palette black)" if c == IGNORE_BGR else "-> UNMAPPED -> ignore"))
    print(f"  BGR{str(c):<16} {n/sum(colour_hits.values())*100:6.2f}%  {tag_c}")

tot = class_px.sum()
print("\nclass pixel share (sample):")
for i, nm in enumerate(CODE_NAMES):
    print(f"  {i if i < 6 else 255:>3} {nm:<14} {class_px[i]/tot*100:6.2f}%")
print(f"\nunmapped pixel share: {100*unmapped/total:.6f}%")

# Orientation guard: Residential must dominate River. If this trips, your channel order is flipped.
assert class_px[0] > class_px[2] * 3, (
    "Residential <= River -- BGR/RGB channel order is probably inverted. STOP and fix before training.")
assert unmapped == 0, f"{unmapped} pixels ({100*unmapped/total:.6f}%) are off-palette -- fix the masks first"
print("channel-order check PASSED | LUT is exact")

## 6. One-time repack to a 512x512 cache

Byte-identical to the segmentation notebooks' repack, so if you already built that cache for the
U-Net or SegFormer runs, set `PREBUILT_CACHE` to it and this cell is a no-op.

Your tiles are **2345x2351 on disk (~10 MB each)**. Decoding 5.5 MP PNGs on Kaggle's 4 vCPUs would
leave the T4 idle, and here it would also make the one-time VAE encoding pass in section 11 far
slower than the training it feeds.

The repack does the resize **once**:
- **Images**: `INTER_AREA` (the correct kernel for 4.6x downsampling; bilinear aliases thin roads).
- **Masks**: colour -> class index at **full resolution first**, then `INTER_NEAREST` to 512.
  Resizing the RGB mask first would interpolate between class colours and manufacture phantom
  classes at every polygon edge — and a *generative* model would then learn to paint those artifacts.

Result: ~700 MB instead of 13 GB, and an effective GSD of 300 m / 512 px = **0.586 m/px**, which is
the figure to report. **Run once, then `Save Version` -> add the output as a dataset -> set
`PREBUILT_CACHE`** so later sessions skip it (~15-20 min saved per session).


In [ ]:
from concurrent.futures import ProcessPoolExecutor

CACHE  = Path(cfg.PREBUILT_CACHE) if cfg.PREBUILT_CACHE else Path(cfg.CACHE_DIR)
MARKER = CACHE / "_complete.json"

def _repack_one(rec):
    _, _, iname, mname = rec
    img = cv2.imread(str(IMG_DIR / iname), cv2.IMREAD_COLOR)
    img = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_AREA)
    msk = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)
    keys = pack_bgr(msk)
    lab  = LUT[keys]                                            # full-res -> class index
    bad  = int((~KNOWN[keys]).sum())
    lab  = cv2.resize(lab, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(str(CACHE / "images" / iname), img, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    cv2.imwrite(str(CACHE / "labels" / mname), lab, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    return np.bincount(np.where(lab == cfg.IGNORE_INDEX, 6, lab).ravel(), minlength=7), bad, keys.size

if MARKER.exists():
    meta = json.load(open(MARKER))
    print(f"cache present at {CACHE} ({meta['n']} pairs) -- skipping repack")
else:
    assert not cfg.PREBUILT_CACHE, f"PREBUILT_CACHE set but {MARKER} missing"
    (CACHE / "images").mkdir(parents=True, exist_ok=True)
    (CACHE / "labels").mkdir(parents=True, exist_ok=True)
    t0 = time.time(); counts, bad_px, tot_px = {}, 0, 0
    with ProcessPoolExecutor(max_workers=4) as ex:
        for rec, (b, bad, n) in zip(records, ex.map(_repack_one, records, chunksize=8)):
            counts[rec[2]] = b.tolist(); bad_px += bad; tot_px += n
            if len(counts) % 200 == 0:
                el = time.time() - t0
                print(f"  {len(counts)}/{len(records)}  {el:.0f}s  "
                      f"eta {el/len(counts)*(len(records)-len(counts)):.0f}s")
    json.dump({"n": len(counts), "counts": counts, "size": cfg.IMG_SIZE,
               "unmapped_px": bad_px, "total_px": tot_px,
               "unmapped_pct": 100.0 * bad_px / max(tot_px, 1)}, open(MARKER, "w"))
    print(f"repacked {len(counts)} pairs in {time.time()-t0:.0f}s")

meta = json.load(open(MARKER))
PIX_COUNTS = {k: np.array(v, dtype=np.int64) for k, v in meta["counts"].items()}
print(f"LUT unmapped over the FULL dataset: {meta.get('unmapped_pct', float('nan')):.6f}% "
      f"({meta.get('unmapped_px', 0)} px)")
assert meta.get("unmapped_pct", 0.0) == 0.0, "off-palette pixels present in the full dataset"
for sub in ("images", "labels"):
    print(f"  {sub}: {len(list((CACHE / sub).glob('*.png')))} files")
print(f"cache size: {sum(f.stat().st_size for f in CACHE.rglob('*.png'))/1e6:.1f} MB")

## 7. Load the cache into RAM

1224 x (512x512x3 + 512x512) uint8 = **~1.3 GB**, comfortably inside Kaggle's ~13 GB. Holding it
resident removes disk I/O from the VAE encoding pass and from the training loop.

Unlike the segmentation notebooks, the **images are only needed once** — section 11 encodes them to
latents and nothing after that ever reads a pixel of a real training image except the qualitative
figures, which re-read from disk. `FREE_IMAGES_AFTER_CACHE` therefore drops them after encoding,
which is what keeps the ~230 MB latent cache and the model comfortably inside RAM together.

The assert is the same one as the other notebooks: labels must contain nothing but `{0..5, 255}`.
An unexpected value here would become a phantom eighth condition channel rather than a device-side
assert.


In [ ]:
RAM = {}
if cfg.PRELOAD_RAM:
    t0 = time.time()
    for i, (_, _, iname, mname) in enumerate(records):
        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        RAM[iname] = (img, lab)
        if (i + 1) % 400 == 0: print(f"  loaded {i+1}/{len(records)}")
    mb = sum(a.nbytes + b.nbytes for a, b in RAM.values()) / 1e6
    print(f"preloaded {len(RAM)} pairs, {mb:.0f} MB, {time.time()-t0:.0f}s")
    u = np.unique(np.concatenate([b[::32, ::32].ravel() for _, b in RAM.values()]))
    assert set(u.tolist()) <= set(range(cfg.N_CLASSES)) | {cfg.IGNORE_INDEX}, \
        f"unexpected label values: {u}"
    print("label value check PASSED:", u.tolist())

GLOBAL_PX = np.sum(list(PIX_COUNTS.values()), axis=0)
print("full-dataset class share (%):",
      {CODE_NAMES[i]: round(100 * GLOBAL_PX[i] / GLOBAL_PX.sum(), 2) for i in range(7)})

## 8. Splitter — fold 0 of the sequence-block CV

`block_split` is **copied verbatim** from the segmentation notebooks and from Stage 1. It must be,
because all three run the identical function and have to agree on which tiles exist. Do not
reimplement it, do not tidy it, do not change the buffer.

For fold 0 that resolves to: sort the 408 base ids ascending, cut into 3 contiguous blocks, block 0
(positions 0..135) is validation, blocks 1+2 are train, and the `BLOCK_BUFFER=2` tiles immediately
after the validation block are dropped from **train** — never from val. Result:
**270 train tiles / 136 val tiles / 2 buffered out**, or 810 train images once the three flip
variants of each train tile are pulled in.

### The leakage rule
The generator is fitted on `tr_recs` and nothing else. `va_recs` is used in exactly two places, both
gradient-free: the fixed-noise eps-MSE monitor in section 17, and the held-out evaluation in
section 19 — which is the whole point of holding them out. Set `VAL_MONITOR=False` if you would
rather the val tiles were not touched during training at all; the cost is that model selection falls
back to the last step instead of the best one.

The split is written to `fold0_split.json` with a SHA1 of the train id list. **Paste Stage 1's
printed `train_sha1` into `cfg.STAGE1_TRAIN_SHA1`** and this cell turns the "do we agree on the
split" question into an assert instead of a conversation.


In [ ]:
def block_split(base_ids: List[int], fold: int, n_folds: int, buffer: int):
    n = len(base_ids)
    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]
    lo, hi = edges[fold], edges[fold + 1]
    val = base_ids[lo:hi]
    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]
    return train, val

assert cfg.SPLIT_MODE == "block" and cfg.FOLD == 0, "stage 2 is defined on block fold 0 only"
tr_ids, va_ids = block_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.BLOCK_BUFFER)

def expand(ids): return [r for i in ids for r in groups[i]]
tr_recs, va_recs = expand(tr_ids), expand(va_ids)

assert not (set(tr_ids) & set(va_ids)), "base-tile leakage between train and val"
buffered = sorted(set(BASE_IDS) - set(tr_ids) - set(va_ids))
split_sha1 = hashlib.sha1(",".join(map(str, sorted(tr_ids))).encode()).hexdigest()

SPLIT = {"split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "n_folds": cfg.N_FOLDS,
         "block_buffer": cfg.BLOCK_BUFFER, "rule": "block_split(), verbatim from the part-1 notebooks",
         "train_ids": sorted(tr_ids), "val_ids": sorted(va_ids), "buffered_ids": buffered,
         "train_sha1": split_sha1}
json.dump(SPLIT, open(f"{cfg.OUT_DIR}/fold0_split.json", "w"), indent=2)

print(f"{cfg.SPLIT_MODE} fold {cfg.FOLD}: {len(tr_ids)} train tiles / {len(va_ids)} val tiles "
      f"({len(buffered)} buffered out: {buffered})")
print(f"  -> {len(tr_recs)} train pairs / {len(va_recs)} val pairs")
print(f"  val id range: {min(va_ids)}..{max(va_ids)} | train id range: {min(tr_ids)}..{max(tr_ids)}")
print(f"  train_sha1: {split_sha1}   <-- must match Stage 1")
if cfg.STAGE1_TRAIN_SHA1:
    assert split_sha1 == cfg.STAGE1_TRAIN_SHA1, (
        f"split disagreement with Stage 1\n  stage1: {cfg.STAGE1_TRAIN_SHA1}\n  stage2: {split_sha1}")
    print("  Stage 1 agreement: ASSERTED")
print(f"  saved -> {cfg.OUT_DIR}/fold0_split.json")

## 9. The condition tensor — one-hot, and why the old pipeline could not work

This replaces the loss section of the segmentation notebooks and the binary-coding section of
Stage 1, and it is the single most consequential choice in this stage.

### What went wrong before
The old pipeline fed the **project's own BGR palette** straight into `control_v11p_sd15_seg`. That
ControlNet was trained on ADE20K segmentation maps, where the colour *is* the class label: it has
learned that `(180,120,120)` means building and `(4,200,3)` means tree. Our palette
— `(128,0,0)`, `(0,128,0)`, `(0,0,128)` — is a different code entirely, so the branch received an
image made of colours it had never been trained to interpret. The mask was not weakly followed; it
carried no class information at all.

### The false-colour metric
A palette is not just a lookup, it imposes a **geometry** on the classes, and ours imposes a
nonsensical one. The cell below prints the pairwise RGB distances. Road `(0,128,0)` sits 128 away
from Forest `(0,128,128)` but 181 away from Residential `(128,0,0)` — so a convolutional encoder is
being told, before it learns anything, that roads resemble forests more than they resemble
buildings. One-hot removes that: every class is exactly `sqrt(2)` from every other, which is the
only honest statement about a nominal variable.

### The three modes
| `COND_MODE` | Channels | What it is |
|---|---|---|
| `onehot` | 7 | 6 classes + ignore, `{0,1}`. The method. Requires the conv swap in section 12. |
| `palette` | 3 | the project's own BGR palette, reversed to RGB. **Ablation A** — the strawman. |
| `ade20k` | 3 | our classes remapped to the nearest ADE20K colours. **Ablations B and C.** |

The ADE20K remap uses the palette that `control_v11p_sd15_seg` was actually trained on: Residential
-> building `(180,120,120)`, Road -> road `(140,140,140)`, River -> water `(61,230,250)`, Forest ->
tree `(4,200,3)`, UnusedLand -> earth `(120,120,70)`, Agricultural -> grass `(4,250,7)`. That row is
the fair comparison — it is the best you can do with a frozen off-the-shelf branch, and it is what
makes the case that the trained branch is doing something the remap cannot.

### Canny
The canny condition is the **class-boundary map of the mask itself** — an exact 1-pixel edge image,
not `cv2.Canny` on a rendering, so there are no thresholds to tune and no texture to leak. It is
used at sampling time only (section 17), never in training.


In [ ]:
PALETTE_RGB = np.zeros((256, 3), np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    PALETTE_RGB[idx] = bgr[::-1]                       # BGR table -> RGB
PALETTE_RGB[cfg.IGNORE_CODE] = (0, 0, 0)               # channel 6, inside the model
PALETTE_RGB[cfg.IGNORE_INDEX] = (0, 0, 0)              # 255, on disk

# ADE20K palette entries that control_v11p_sd15_seg was actually trained on.
ADE20K_MAP = {0: (180, 120, 120),   # Residential  -> building;edifice
              1: (140, 140, 140),   # Road         -> road;route
              2: (61, 230, 250),    # River        -> water
              3: (4, 200, 3),       # Forest       -> tree
              4: (120, 120, 70),    # UnusedLand   -> earth;ground
              5: (4, 250, 7)}       # Agricultural -> grass
ADE20K_RGB = np.zeros((256, 3), np.uint8)
for i, c in ADE20K_MAP.items(): ADE20K_RGB[i] = c
ADE20K_RGB[cfg.IGNORE_CODE] = (0, 0, 0); ADE20K_RGB[cfg.IGNORE_INDEX] = (0, 0, 0)

EYE7 = np.eye(cfg.N_COND_CH, dtype=np.float32)

def to_code(lab: np.ndarray) -> np.ndarray:
    # on-disk {0..5, 255} -> condition codes {0..6}; 6 accepted as an alias for ignore
    lab = np.where(lab == cfg.IGNORE_INDEX, cfg.IGNORE_CODE, lab)
    return lab.astype(np.uint8)

def cond_from_index(lab: np.ndarray, mode: Optional[str] = None) -> np.ndarray:
    # (H, W) class indices -> (C, H, W) float32 in [0, 1]; ControlNet expects that range
    mode = mode or cfg.COND_MODE
    code = to_code(lab)
    if mode == "onehot":
        return np.ascontiguousarray(EYE7[code].transpose(2, 0, 1))
    tbl = PALETTE_RGB if mode == "palette" else ADE20K_RGB
    return np.ascontiguousarray(tbl[code].transpose(2, 0, 1).astype(np.float32) / 255.0)

def canny_from_index(lab: np.ndarray) -> np.ndarray:
    # exact 1-px class-boundary map, replicated to 3 channels, for the frozen canny branch
    code = to_code(lab)
    e = np.zeros(code.shape, np.uint8)
    dh = (code[:, 1:] != code[:, :-1]); dv = (code[1:, :] != code[:-1, :])
    e[:, 1:] |= dh; e[:, :-1] |= dh; e[1:, :] |= dv; e[:-1, :] |= dv
    return np.ascontiguousarray(np.repeat(e[None].astype(np.float32), 3, axis=0))

# ---- the false-colour metric, printed rather than asserted ----
P = PALETTE_RGB[:cfg.N_COND_CH].astype(np.float32)
D_pal = np.linalg.norm(P[:, None] - P[None], axis=-1)
print("pairwise RGB distance under the PROJECT palette (what the old pipeline fed the branch):")
print("            " + "".join(f"{n[:6]:>8}" for n in CODE_NAMES))
for i, n in enumerate(CODE_NAMES):
    print(f"  {n:<12}" + "".join(f"{D_pal[i, j]:8.0f}" for j in range(cfg.N_COND_CH)))
print(f"\n  Road-Forest {D_pal[1,3]:.0f}  vs  Road-Residential {D_pal[1,0]:.0f}"
      "   <- roads are told they resemble forests")
print(f"  under one-hot every pair is {math.sqrt(2):.4f} apart, which is the only honest "
      "statement about a nominal variable\n")

# ---- round trip: one-hot must be exactly invertible ----
_lab = np.random.RandomState(cfg.SEED).choice(
    np.array([0, 1, 2, 3, 4, 5, cfg.IGNORE_INDEX], np.uint8), (64, 64))
_c = cond_from_index(_lab, "onehot")
assert (_c.argmax(0).astype(np.uint8) == to_code(_lab)).all(), "one-hot round trip is not exact"
assert _c.sum(0).min() == 1.0 and _c.max() == 1.0, "one-hot is not a partition of unity"
print(f"condition mode: {cfg.COND_MODE} -> {COND_CH} channels | one-hot round trip exact | "
      f"canny boundary map {'ON at sampling' if cfg.USE_CANNY else 'OFF'}")

## 10. Captions

Short templated captions built from the mask's own class proportions. Deliberately simple: the mask
does the real work, and a caption that tries to describe layout would compete with the control
branch rather than support it. **No LLM planner** — the old pipeline's 4-step prompt planner is
demoted to the application section, because a prompt that varies per scene in ways the mask does not
would make the ablation table uninterpretable.

Construction: take the class shares, drop anything under 2%, keep the **top three by share**, then
**re-admit Road and River** whenever they cover at least 0.5%. That second step is not cosmetic. Road
is 2.28% of pixels and River is 0.57%, so a plain share-ordered cut throws away exactly the two
classes this project exists to improve — the caption would say "dense residential rooftops,
agricultural fields, forest" for a tile whose river is the only interesting thing in it. Quantifiers
(`dense` above 35%, `scattered` below 10%) are applied to the area classes only, because "dense
river" is not English. The result is a small closed vocabulary rather than 1224 distinct strings,
which is what makes the text embeddings cheap enough to cache outright in section 11.

`CAPTION_DROP=0.1` replaces the caption with `""` one time in ten during training. That is not
cosmetic: it is what gives classifier-free guidance an unconditional branch to interpolate away
from, and it is the same 10% dropout SD 1.5 itself was trained with.


In [ ]:
NOUN = {0: "residential rooftops", 1: "road network", 2: "river",
        3: "forest", 4: "bare unused land", 5: "agricultural fields"}
AREA_CLASSES = {0, 3, 4, 5}          # quantifiers only make sense for these
THIN_CLASSES = (1, 2)                # Road, River -- always mentioned when present
THIN_FLOOR   = 0.005                 # River is 0.57% of pixels globally; 2% would erase it

def caption_from_index(lab: np.ndarray) -> str:
    code = to_code(lab)
    h = np.bincount(code.ravel(), minlength=cfg.N_COND_CH)[:cfg.N_CLASSES].astype(np.float64)
    s = h / max(h.sum(), 1.0)
    keep = [int(c) for c in np.argsort(-s) if s[c] >= 0.02][:3]
    for c in THIN_CLASSES:                       # re-admit the rare classes a share cut would drop
        if s[c] >= THIN_FLOOR and c not in keep: keep.append(c)
    parts = []
    for c in sorted(keep, key=lambda c: -s[c]):
        q = ""
        if c in AREA_CLASSES:
            q = "dense " if s[c] >= 0.35 else ("scattered " if s[c] < 0.10 else "")
        parts.append(q + NOUN[c])
    return "aerial satellite view of Kathmandu Valley, " + ", ".join(parts)

def _label_of(rec):
    return RAM[rec[2]][1] if cfg.PRELOAD_RAM else cv2.imread(
        str(CACHE / "labels" / rec[3]), cv2.IMREAD_GRAYSCALE)

CAPTIONS = {r[2]: caption_from_index(_label_of(r)) for r in records}
VOCAB = sorted(set(CAPTIONS.values()) | {""})

print(f"{len(CAPTIONS)} captions over {len(VOCAB)} distinct strings "
      f"(incl. the empty caption used by CFG)")
_freq = {}
for c in CAPTIONS.values(): _freq[c] = _freq.get(c, 0) + 1
for c, n in sorted(_freq.items(), key=lambda kv: -kv[1])[:6]:
    print(f"  {n:>4}x  {c}")
print(f"  ... {max(0, len(VOCAB)-7)} more")
_tok_lens = [len(c.split()) for c in VOCAB if c]
print(f"caption length: {min(_tok_lens)}-{max(_tok_lens)} words, well inside CLIP's 77 tokens")

## 11. Frozen backbone — cached latents and text embeddings, dataset and dataloaders

The VAE and the text encoder never train, so running them inside the training loop would burn
throughput on a computation whose answer never changes. Both are evaluated **once**, here, and the
results are held in RAM.

### Latents
Every training image is encoded to a 4x64x64 Gaussian, and the **mean and std are cached rather than
a single sample**, so the per-step latent is still drawn afresh — the same stochasticity SD training
normally has, at no extra cost. All four rot90 orientations are encoded separately rather than
rotating the latent, because a convolutional VAE is only approximately rotation-equivariant and
approximating it here would put a systematic error into three quarters of the training set. The
whole cache is 810 x 4 + 408 latents at 2 x 4x64x64 fp16 = **~230 MB**.

The VAE is kept in **fp32**. SD 1.5's VAE is the well-known fp16 overflow case, and 335 MB of VRAM
is a cheap price for never having to debug a NaN that appears once every few hundred decodes.

### Text
The caption vocabulary from section 10 is small, so every distinct caption is embedded once into a
77x768 fp16 tensor and looked up by string. The text encoder stays resident afterwards (246 MB in
fp16) because section 21 has to caption masks it has never seen.

### Augmentation
**No hflip/vflip** — those are already materialised in the 1224 files. On-the-fly augmentation is
**rot90 (k in 0..3) only**, which together with the baked-in flips completes the dihedral group D4,
the natural symmetry group for nadir overhead imagery. rot90 is exact — no resampling, so 9 px roads
and 13 px rivers stay crisp in both the image and the condition. The photometric jitter the
segmentation notebooks use is deliberately **not** applied: it would teach the generator that
Kathmandu comes in several exposures, and colour realism is exactly what KID measures.


In [ ]:
from diffusers import AutoencoderKL, UNet2DConditionModel, ControlNetModel
from diffusers import DDPMScheduler, DDIMScheduler
from transformers import CLIPTextModel, CLIPTokenizer

tokenizer    = CLIPTokenizer.from_pretrained(cfg.SD_MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(cfg.SD_MODEL_ID, subfolder="text_encoder",
                                             torch_dtype=torch.float16).cuda().eval()
vae          = AutoencoderKL.from_pretrained(cfg.SD_MODEL_ID, subfolder="vae").cuda().eval()  # fp32
text_encoder.requires_grad_(False); vae.requires_grad_(False)

noise_scheduler = DDPMScheduler.from_pretrained(cfg.SD_MODEL_ID, subfolder="scheduler")
SCALING = float(vae.config.scaling_factor)
print(f"VAE scaling {SCALING} | train scheduler: {noise_scheduler.config.beta_schedule}, "
      f"T={noise_scheduler.config.num_train_timesteps}, "
      f"pred={noise_scheduler.config.prediction_type}")

@torch.no_grad()
def embed_captions(caps: List[str]) -> torch.Tensor:
    tok = tokenizer(list(caps), padding="max_length", max_length=tokenizer.model_max_length,
                    truncation=True, return_tensors="pt")
    return text_encoder(tok.input_ids.cuda())[0]          # (B, 77, 768) fp16

t0 = time.time()
TEXTEMB: Dict[str, torch.Tensor] = {}
for i in range(0, len(VOCAB), 16):
    chunk = VOCAB[i:i + 16]
    for c, v in zip(chunk, embed_captions(chunk).cpu()):
        TEXTEMB[c] = v
print(f"cached {len(TEXTEMB)} text embeddings in {time.time()-t0:.0f}s "
      f"({sum(v.numel()*2 for v in TEXTEMB.values())/1e6:.0f} MB)")

@torch.no_grad()
def encode_images(imgs_bgr: List[np.ndarray]):
    x = np.stack([cv2.cvtColor(a, cv2.COLOR_BGR2RGB) for a in imgs_bgr]).astype(np.float32)
    x = torch.from_numpy((x / 127.5 - 1.0).transpose(0, 3, 1, 2)).cuda()
    d = vae.encode(x).latent_dist
    return d.mean.half().cpu(), d.std.half().cpu()        # scaling is applied at sample time

ROTS = range(4) if cfg.AUG_ROT90 else [0]
JOBS = [(r[2], k) for r in tr_recs for k in ROTS] + [(r[2], 0) for r in va_recs]
LAT: Dict[Tuple[str, int], Tuple[torch.Tensor, torch.Tensor]] = {}

t0 = time.time()
for s in range(0, len(JOBS), 8):
    batch = JOBS[s:s + 8]
    imgs = []
    for iname, k in batch:
        im = RAM[iname][0] if cfg.PRELOAD_RAM else cv2.imread(str(CACHE / "images" / iname),
                                                              cv2.IMREAD_COLOR)
        imgs.append(np.ascontiguousarray(np.rot90(im, k)) if k else im)
    mu, sd = encode_images(imgs)
    for (iname, k), m_, s_ in zip(batch, mu, sd):
        LAT[(iname, k)] = (m_.clone(), s_.clone())
    if (s // 8) % 60 == 0:
        el = time.time() - t0
        print(f"  encoded {min(s+8, len(JOBS))}/{len(JOBS)}  {el:.0f}s  "
              f"eta {el/max(s+8,1)*(len(JOBS)-s-8):.0f}s")
lat_mb = sum(a.numel() * 2 + b.numel() * 2 for a, b in LAT.values()) / 1e6
print(f"cached {len(LAT)} latents ({lat_mb:.0f} MB) in {time.time()-t0:.0f}s")

if cfg.PRELOAD_RAM and cfg.FREE_IMAGES_AFTER_CACHE:
    for k in list(RAM): RAM[k] = (None, RAM[k][1])       # labels stay, pixels go
    gc.collect()
    print("released cached images from RAM (labels retained)")

class MaskToImage(Dataset):
    def __init__(self, recs, train: bool):
        self.recs, self.train, self.epoch = recs, train, 0

    def __len__(self): return len(self.recs)

    def _label(self, rec):
        lab = RAM[rec[2]][1] if cfg.PRELOAD_RAM else cv2.imread(
            str(CACHE / "labels" / rec[3]), cv2.IMREAD_GRAYSCALE)
        return lab

    def __getitem__(self, i):
        rec = self.recs[i]; iname = rec[2]
        rs = np.random.RandomState((cfg.SEED * 100003 + self.epoch * 9973 + i) % (2 ** 31))
        k = rs.randint(4) if (self.train and cfg.AUG_ROT90) else 0
        mu, sd = LAT[(iname, k)]
        eps = torch.from_numpy(rs.standard_normal(tuple(mu.shape)).astype(np.float32))
        lat = (mu.float() + sd.float() * eps) * SCALING     # fresh draw, cached distribution
        lab = self._label(rec)
        if k: lab = np.ascontiguousarray(np.rot90(lab, k))
        cond = torch.from_numpy(cond_from_index(lab))
        cap = CAPTIONS[iname]
        if self.train and rs.rand() < cfg.CAPTION_DROP: cap = ""
        return lat, cond, TEXTEMB[cap]

train_ds, val_ds = MaskToImage(tr_recs, True), MaskToImage(va_recs, False)
# persistent_workers is OFF on the train loader on purpose: with it on, the worker processes
# keep their own copy of train_ds and never see `train_ds.epoch` advance, so every image would
# get the same rot90 forever. Re-forking costs ~1 s per pass and buys real augmentation variety.
train_ld = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=True,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True, persistent_workers=False,
                      worker_init_fn=worker_init_fn, generator=g)
val_ld   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True, persistent_workers=True,
                      worker_init_fn=worker_init_fn)

_lat, _cond, _emb = next(iter(train_ld))
assert _lat.shape[1:] == (4, cfg.LATENT_SIZE, cfg.LATENT_SIZE), _lat.shape
assert _cond.shape[1:] == (COND_CH, cfg.IMG_SIZE, cfg.IMG_SIZE), _cond.shape
print(f"batch: latents {tuple(_lat.shape)} | condition {tuple(_cond.shape)} "
      f"[{_cond.min():.2f}, {_cond.max():.2f}] | text {tuple(_emb.shape)}")
print(f"{len(train_ds)} train pairs / {len(val_ds)} val pairs | "
      f"{len(train_ld)//cfg.GRAD_ACCUM} optimizer steps per pass | "
      f"{cfg.MAX_STEPS/(len(train_ld)//cfg.GRAD_ACCUM):.0f} passes planned")

## 12. Model — frozen SD1.5, LoRA on the attention projections, warm-started control branch

### The three pieces
| Piece | State | Params | LR |
|---|---|---|---|
| SD 1.5 UNet | **frozen, fp16** | 860 M | — |
| LoRA on `to_q`/`to_k`/`to_v`/`to_out.0` | trained, fp32 | ~1.6 M at r=16 | 1e-4 |
| control branch, warm-started from `control_v11p_sd15_seg` | trained, fp32 | 361 M | 1e-5 |

The two learning rates are two orders of magnitude apart on purpose. LoRA is learning a *new* thing
— what Kathmandu looks like from above — from a zero initialization, so it needs a real learning
rate. The control branch already knows how to turn a layout into spatial guidance; it is being
adapted, not taught, and 1e-5 is the rate at which a 361 M-parameter branch can be nudged on 810
images without washing out the pretraining that made warm-starting worth doing.

### The conditioning conv swap
`ControlNetConditioningEmbedding.conv_in` is a 3->16 convolution over the RGB condition. For
`COND_MODE="onehot"` it is replaced by a 7->16 convolution and **zero-initialized**, so the branch
starts from a stable point rather than from a random projection of seven channels it has never seen.

One honest consequence, worth stating in the paper rather than discovering later: because the
downstream convolutions of the embedding keep their pretrained biases, a zeroed `conv_in` makes the
embedding output a spatially *constant* map, not a zero one. The branch therefore begins by ignoring
the mask entirely and has to learn to use it — which is the intended behaviour and is why the layout
fidelity number in section 19 is near chance for the first few hundred steps. `COND_CONV_INIT`
offers an alternative: `palette_project` seeds the new conv with the pretrained RGB filters
projected through the ADE20K colours of section 9, so channel *c* starts out producing exactly the
response the pretrained branch would have produced for that class's ADE20K colour. It converges
faster in practice; `zeros` is the default because it is the plainer claim.

### Memory — why this fits on a 16 GB T4, and what is doing the work
| Item | VRAM |
|---|---|
| UNet fp16, frozen | 1.7 GB |
| control branch fp32, trained | 1.4 GB |
| its gradients fp32 | 1.4 GB |
| 8-bit AdamW state (2 moments, uint8) | 0.7 GB |
| VAE fp32 + text encoder fp16 | 0.6 GB |
| activations at `BATCH_SIZE=2` with checkpointing | ~2-3 GB |

That is roughly **8-10 GB**, measured for real in section 14 before you commit to a long run. The
levers, in the order to reach for them:

| Lever | Effect | Cost |
|---|---|---|
| `BATCH_SIZE=2`, `GRAD_ACCUM=8` | **effective batch stays 16** | none — identical gradient up to fp16 rounding |
| `GRAD_CHECKPOINT=True` on both branches | ~3x less activation memory | ~30% slower per step |
| `USE_8BIT_ADAM=True` | 0.7 GB instead of 2.9 GB of optimizer state | negligible quality effect at this LR |

**Effective batch is protocol, micro-batch is not.** If you OOM, set `BATCH_SIZE=1`,
`GRAD_ACCUM=16`; do not lower `BATCH_SIZE` on its own, because that changes the optimization and
makes this run incomparable to any rerun.

### The backward guard
Gradient checkpointing that detaches the graph trains nothing and never raises. The guard at the end
of this cell runs one real micro-step and asserts that gradient actually reaches every trainable
group — five seconds here instead of six hours of a run that was optimizing nothing.


In [ ]:
unet = UNet2DConditionModel.from_pretrained(cfg.SD_MODEL_ID, subfolder="unet",
                                            torch_dtype=torch.float16).cuda()
unet.requires_grad_(False)

controlnet = ControlNetModel.from_pretrained(cfg.CONTROLNET_ID, torch_dtype=torch.float32).cuda()

# ---- swap the conditioning input conv for the one-hot condition ----
if cfg.COND_MODE == "onehot":
    old = controlnet.controlnet_cond_embedding.conv_in
    new = nn.Conv2d(COND_CH, old.out_channels, kernel_size=old.kernel_size,
                    stride=old.stride, padding=old.padding)
    with torch.no_grad():
        if cfg.COND_CONV_INIT == "palette_project":
            # channel c starts out producing the pretrained response to that class's ADE20K colour
            Pc = torch.tensor([[*ADE20K_RGB[i]] for i in range(COND_CH)],
                              dtype=torch.float32) / 255.0            # (7, 3)
            new.weight.copy_(torch.einsum("okhw,ck->ochw", old.weight.data.float(), Pc))
            new.bias.copy_(old.bias.data.float())
        else:
            new.weight.zero_(); new.bias.zero_()
    controlnet.controlnet_cond_embedding.conv_in = new.to("cuda", torch.float32)
    controlnet.register_to_config(conditioning_channels=COND_CH)
    print(f"conditioning conv replaced: 3 -> {COND_CH} channels, init={cfg.COND_CONV_INIT}")
else:
    print(f"conditioning conv left at 3 channels (COND_MODE={cfg.COND_MODE})")

controlnet.requires_grad_(bool(cfg.TRAIN_CONTROLNET))

# ---- LoRA on the UNet attention projections ----
if cfg.USE_LORA:
    from peft import LoraConfig
    unet.add_adapter(LoraConfig(r=cfg.LORA_RANK, lora_alpha=cfg.LORA_ALPHA,
                                lora_dropout=cfg.LORA_DROPOUT, init_lora_weights="gaussian",
                                target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
    for p in unet.parameters():
        if p.requires_grad: p.data = p.data.float()     # trainable params in fp32, base stays fp16

if cfg.GRAD_CHECKPOINT and TRAINABLE:
    if cfg.USE_LORA: unet.enable_gradient_checkpointing()
    if cfg.TRAIN_CONTROLNET: controlnet.enable_gradient_checkpointing()

LORA_PARAMS = [p for p in unet.parameters() if p.requires_grad]
CN_PARAMS   = [p for p in controlnet.parameters() if p.requires_grad]

def build_optimizer():
    groups = []
    if LORA_PARAMS:
        groups.append({"params": LORA_PARAMS, "lr": cfg.LR_LORA, "weight_decay": 0.0,
                       "name": "lora"})               # LoRA is conventionally trained without decay
    if CN_PARAMS:
        dec, nod = [], []
        for n, p in controlnet.named_parameters():
            if not p.requires_grad: continue
            (nod if (p.ndim <= 1 or n.endswith(".bias")) else dec).append(p)
        nwd = 0.0 if cfg.NO_DECAY_ON_NORM_BIAS else cfg.WEIGHT_DECAY
        groups += [{"params": dec, "lr": cfg.LR_CONTROL, "weight_decay": cfg.WEIGHT_DECAY,
                    "name": "control_decay"},
                   {"params": nod, "lr": cfg.LR_CONTROL, "weight_decay": nwd,
                    "name": "control_nodecay"}]
    if not groups:                                     # ablations A and B train nothing
        return None, None
    if cfg.USE_8BIT_ADAM and HAS_BNB:
        opt = bnb.optim.AdamW8bit(groups, betas=(0.9, 0.999), eps=1e-8)
    else:
        opt = torch.optim.AdamW(groups, betas=(0.9, 0.999), eps=1e-8)
        if cfg.USE_8BIT_ADAM: print("  8-bit AdamW requested but bitsandbytes is unavailable "
                                    "-- using torch.AdamW (+~2.2 GB VRAM)")
    def lr_lambda(step):
        if step < cfg.WARMUP_STEPS: return (step + 1) / cfg.WARMUP_STEPS
        prog = (step - cfg.WARMUP_STEPS) / max(1, cfg.MAX_STEPS - cfg.WARMUP_STEPS)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

optimizer, scheduler = build_optimizer()
scaler = torch.cuda.amp.GradScaler(enabled=cfg.AMP)

def cn_unet(latents, t, emb, cond, cond_scale=None, extra=None):
    # control branch -> residuals -> frozen UNet. `extra` is an optional list of
    # (controlnet, cond, scale) whose residuals are summed in (used for canny at sampling).
    cs = cfg.COND_SCALE if cond_scale is None else cond_scale
    down, mid = controlnet(latents, t, encoder_hidden_states=emb, controlnet_cond=cond,
                           conditioning_scale=cs, return_dict=False)
    down = list(down)
    for net_x, cond_x, sc_x in (extra or []):
        d2, m2 = net_x(latents, t, encoder_hidden_states=emb, controlnet_cond=cond_x,
                       conditioning_scale=sc_x, return_dict=False)
        down = [a + b.to(a.dtype) for a, b in zip(down, d2)]
        mid = mid + m2.to(mid.dtype)
    down = [d.to(unet.dtype) for d in down]
    return unet(latents, t, encoder_hidden_states=emb,
                down_block_additional_residuals=down,
                mid_block_additional_residual=mid.to(unet.dtype)).sample

n_lora = sum(p.numel() for p in LORA_PARAMS)
n_cn   = sum(p.numel() for p in CN_PARAMS)
n_frozen = sum(p.numel() for p in unet.parameters() if not p.requires_grad)
ARCH_SIG = (f"sd15ctrl|{cfg.SD_MODEL_ID}|{cfg.CONTROLNET_ID}|cond{cfg.COND_MODE}:{COND_CH}|"
            f"init{cfg.COND_CONV_INIT}|lora{cfg.USE_LORA}:r{cfg.LORA_RANK}a{cfg.LORA_ALPHA}|"
            f"cntrain{cfg.TRAIN_CONTROLNET}|img{cfg.IMG_SIZE}|p{n_lora}+{n_cn}")

# ---- backward guard: does gradient actually reach every trainable group? ----
if TRAINABLE:
    unet.train(); controlnet.train()
    _l, _c, _e = _lat[:1].cuda(), _cond[:1].cuda(), _emb[:1].cuda()
    _t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (1,), device="cuda")
    _n = torch.randn_like(_l)
    _xt = noise_scheduler.add_noise(_l, _n, _t)
    with torch.cuda.amp.autocast(enabled=cfg.AMP):
        _p = cn_unet(_xt, _t, _e, _c)
    F.mse_loss(_p.float(), _n.float()).backward()
    for nm, ps in [("lora", LORA_PARAMS), ("control", CN_PARAMS)]:
        if not ps: continue
        gn = sum(float(p.grad.norm()) for p in ps if p.grad is not None)
        assert gn > 0, f"no gradient reached the {nm} parameters -- checkpointing detached the graph"
        print(f"  backward guard OK: {nm} grad-norm {gn:.3e}")
    unet.zero_grad(set_to_none=True); controlnet.zero_grad(set_to_none=True)
    del _l, _c, _e, _t, _n, _xt, _p; gc.collect(); torch.cuda.empty_cache()
else:
    print("  ablation trains nothing -- no backward guard, no optimizer")

print(f"params: UNet frozen {n_frozen/1e6:.0f} M | LoRA {n_lora/1e6:.2f} M | "
      f"control branch {n_cn/1e6:.0f} M {'(trained)' if CN_PARAMS else '(frozen)'}")
print(f"optimizer: {type(optimizer).__name__ if optimizer else 'none'} | "
      f"checkpointing {cfg.GRAD_CHECKPOINT and TRAINABLE}")
print("arch:", ARCH_SIG)

## 13. Evaluation module — frozen SegFormer-B0, layout fidelity, clDice, KID/FID

This replaces the segmentation notebooks' metrics module. There is no ground-truth image for a
*generated* scene, so quality is measured three ways and none of them is optional.

### Layout fidelity — the one that decides everything
Generate an image from a held-out real mask, re-segment it with the **frozen part-1 SegFormer-B0**
(`Pranilllllll/segformer-b0-kaggle-ktm-dataset-block-fold0`, inference-only, never fine-tuned here),
and take mIoU against the mask the generator was told to follow. If the generator ignores the mask,
this collapses.

**Read the ratio, not the absolute number.** B0 scores **0.2877 mIoU on real fold-0 val images** —
that is its published number for this fold, and it is the ceiling for this measurement, because a
generated image cannot be easier to segment than the real thing that the same segmenter was trained
on. An absolute floor of 0.30 is therefore essentially unreachable on fold 0 and would fail a
perfectly good generator. This cell computes all three quantities so the comparison is honest:

| Quantity | What it is | What it means |
|---|---|---|
| `miou_gen` | generated image vs its own mask | the measurement |
| `miou_real` | real val image vs the same mask | the ceiling (~0.29 expected) |
| `miou_shuffled` | generated image vs a **different** mask | the null — chance for this class prior |

`layout_ratio = miou_gen / miou_real` is the number to put in the paper, with a pass threshold of
0.60. The shuffled null costs nothing to compute — it re-scores images that already exist — and it
is what rules out the "the segmenter just predicts Residential everywhere" explanation.

### clDice
Same implementation as the part-1 notebooks, on Road and River only. Skeleton-based, so it rewards a
continuous road that is one pixel off over a blob that overlaps it — which is exactly the failure
mode of a generator that paints texture without structure.

### KID and FID
Both from 2048-d pool features of torchvision's ImageNet Inception-v3. **KID is the headline.** With
only a few hundred real validation scenes, FID's estimator bias is large, generator-dependent, and
does not shrink with a better generator, so ranking two models by FID at this N is unsound. KID is
an unbiased MMD estimator: reported as mean +/- sd over `KID_SUBSETS` subsets of `KID_SUBSET_SIZE`,
with both N values printed. Note in the caption that these are torchvision Inception weights, not
the TF-ported ones the original FID paper used, so the absolute values are not comparable to numbers
quoted from other papers — only within this table.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import linalg
from skimage.morphology import skeletonize
from transformers import SegformerForSemanticSegmentation, SegformerConfig
from huggingface_hub import hf_hub_download

IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

# ---- frozen part-1 segmenter: INFERENCE ONLY, never fine-tuned in this notebook ----
def load_segformer(repo_id, token=None):
    ck = torch.load(hf_hub_download(repo_id, "best.pt", token=token), map_location="cuda")
    m = SegformerForSemanticSegmentation(SegformerConfig.from_dict(ck["model_config"])).cuda().eval()
    m.load_state_dict(ck["model_state"])
    m.requires_grad_(False)
    return m, ck.get("cfg", {}), ck.get("metrics", {})

@torch.no_grad()
def segment_rgb01(m, x01, batch=8):
    # x01: (N, 3, H, W) float in [0, 1], RGB. Matches part-1 preprocessing exactly.
    out = []
    for i in range(0, len(x01), batch):
        x = ((x01[i:i + batch].cuda().float() - IMNET_MEAN) / IMNET_STD)
        with torch.cuda.amp.autocast():
            lo = m(pixel_values=x).logits
            lo = F.interpolate(lo, size=x.shape[-2:], mode="bilinear", align_corners=False)
        out.append(lo.float().argmax(1).cpu())
    return torch.cat(out)

# ---- confusion matrix and metrics: copied verbatim from the part-1 notebooks ----
class ConfMat:
    def __init__(self, n): self.n = n; self.m = torch.zeros(n, n, dtype=torch.int64)
    def update(self, pred, tgt):
        k = tgt != cfg.IGNORE_INDEX
        if k.sum() == 0: return
        self.m += torch.bincount(self.n * tgt[k].long() + pred[k].long(),
                                 minlength=self.n ** 2).reshape(self.n, self.n)
    def numpy(self): return self.m.numpy().astype(np.float64)

def metrics_from_cm(cm, names):
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tot = cm.sum()
    iou = tp / np.maximum(tp + fp + fn, 1e-12)
    f1  = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)
    prod = tp / np.maximum(cm.sum(1), 1e-12)
    user = tp / np.maximum(cm.sum(0), 1e-12)
    freq = cm.sum(1) / max(tot, 1e-12)
    po = tp.sum() / max(tot, 1e-12)
    pe = (cm.sum(0) * cm.sum(1)).sum() / max(tot ** 2, 1e-12)
    return {"mIoU": float(np.nanmean(iou)), "mF1": float(np.nanmean(f1)), "OA": float(po),
            "FWIoU": float((freq * iou).sum()), "kappa": float((po - pe) / max(1 - pe, 1e-12)),
            "per_class": {n: {"IoU": float(iou[i]), "F1": float(f1[i]),
                              "producer_acc": float(prod[i]), "user_acc": float(user[i]),
                              "support_px": int(cm.sum(1)[i])} for i, n in enumerate(names)}}

def cldice_pair(pred_bin, gt_bin):
    if gt_bin.sum() == 0 and pred_bin.sum() == 0: return None
    sp = skeletonize(pred_bin.astype(bool)); sg = skeletonize(gt_bin.astype(bool))
    tprec = (sp & gt_bin.astype(bool)).sum() / max(sp.sum(), 1)     # topology precision
    tsens = (sg & pred_bin.astype(bool)).sum() / max(sg.sum(), 1)   # topology sensitivity
    if tprec + tsens == 0: return 0.0
    return float(2 * tprec * tsens / (tprec + tsens))

def layout_scores(preds, labels, names):
    # preds/labels: (N, H, W) int arrays. Returns the full metric dict plus per-class clDice.
    cm = ConfMat(cfg.N_CLASSES)
    for p, t in zip(preds, labels):
        cm.update(torch.as_tensor(p), torch.as_tensor(t))
    out = metrics_from_cm(cm.numpy(), names)
    for c, nm in ((1, "Road"), (2, "River")):
        vals = [v for p, t in zip(preds, labels)
                for v in [cldice_pair(np.asarray(p) == c, np.asarray(t) == c)] if v is not None]
        out[f"clDice_{nm}"] = float(np.mean(vals)) if vals else float("nan")
    return out

# ---- Inception features, FID and KID ----
from torchvision.models import inception_v3, Inception_V3_Weights
_INCEPTION = None

def get_inception():
    global _INCEPTION
    if _INCEPTION is None:
        m = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
        m.fc = nn.Identity()
        _INCEPTION = m.eval().cuda().requires_grad_(False)
    return _INCEPTION

@torch.no_grad()
def inception_feats(x01, batch=16):
    m = get_inception(); out = []
    for i in range(0, len(x01), batch):
        x = x01[i:i + batch].cuda().float()
        x = F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False)
        out.append(m((x - IMNET_MEAN) / IMNET_STD).cpu().numpy())
    return np.concatenate(out).astype(np.float64)

def fid_from_feats(f1, f2, eps=1e-6):
    mu1, mu2 = f1.mean(0), f2.mean(0)
    s1, s2 = np.cov(f1, rowvar=False), np.cov(f2, rowvar=False)
    covmean, _ = linalg.sqrtm(s1.dot(s2), disp=False)
    if not np.isfinite(covmean).all():
        off = np.eye(s1.shape[0]) * eps
        covmean = linalg.sqrtm((s1 + off).dot(s2 + off))
    if np.iscomplexobj(covmean): covmean = covmean.real
    d = mu1 - mu2
    return float(d.dot(d) + np.trace(s1) + np.trace(s2) - 2 * np.trace(covmean))

def kid_from_feats(f1, f2, subset_size=None, subsets=None, seed=0):
    # unbiased MMD^2 with the standard polynomial kernel (degree 3, gamma 1/d, coef 1)
    m = min(subset_size or cfg.KID_SUBSET_SIZE, len(f1), len(f2))
    n_sub = subsets or cfg.KID_SUBSETS
    d = f1.shape[1]; rng = np.random.RandomState(seed); vals = []
    for _ in range(n_sub):
        x = f1[rng.choice(len(f1), m, replace=False)]
        y = f2[rng.choice(len(f2), m, replace=False)]
        kxx = (x.dot(x.T) / d + 1) ** 3
        kyy = (y.dot(y.T) / d + 1) ** 3
        kxy = (x.dot(y.T) / d + 1) ** 3
        vals.append((kxx.sum() - np.trace(kxx)) / (m * (m - 1))
                    + (kyy.sum() - np.trace(kyy)) / (m * (m - 1))
                    - 2 * kxy.mean())
    return float(np.mean(vals)), float(np.std(vals)), int(m), int(n_sub)

# ---- rendering, shared palette with the part-1 figures ----
def colorize(lab): return PALETTE_RGB[to_code(lab)]

LEGEND = ([Patch(color=PALETTE_RGB[i] / 255, label=n) for i, n in enumerate(cfg.CLASS_NAMES)] +
          [Patch(color="black", label="ignore/unannotated")])

def triptych(reals, labels, gens, n=4, title=None, cell=2.6):
    n = min(n, len(gens))
    fig, axes = plt.subplots(n, 3, figsize=(3 * cell, n * cell))
    axes = np.atleast_2d(axes)
    for i in range(n):
        for j, (im, ttl) in enumerate([(reals[i], "real"), (colorize(labels[i]), "mask"),
                                       (gens[i], "generated")]):
            axes[i, j].imshow(im); axes[i, j].axis("off")
            if i == 0: axes[i, j].set_title(ttl, fontsize=10)
    if title: fig.suptitle(title, fontsize=11)
    fig.legend(handles=LEGEND, loc="lower center", ncol=4, fontsize=8, frameon=False)
    plt.tight_layout(rect=[0, 0.05, 1, 0.96])
    return fig

segmodel, segcfg, segmetrics = load_segformer(cfg.SEG_REPO)
print(f"frozen segmenter: {cfg.SEG_REPO} | published val mIoU "
      f"{segmetrics.get('mIoU', float('nan')):.4f} | trainable params "
      f"{sum(p.numel() for p in segmodel.parameters() if p.requires_grad)}")
print("metrics module ready | KID is the headline, FID secondary, N printed with both")

## 14. Efficiency — measured VRAM and seconds per step, before you commit to a long run

This is the cell to read before walking away. It runs `GRAD_ACCUM` real micro-batches plus one real
optimizer step — so the peak it reports **includes** the optimizer state, which a forward-only probe
would miss by 0.7 GB — then rebuilds the optimizer and scheduler so the benchmark leaves no trace
beyond one step taken at warmup learning rate (`LR/WARMUP_STEPS`, i.e. about 5e-7; harmless).

What to expect on a single T4 at `BATCH_SIZE=2` with checkpointing on both branches: roughly
**1.5-2.5 s per micro-batch**, so **12-20 s per optimizer step**, and `MAX_STEPS=3000` lands
somewhere around **10-16 h** — which is one and a bit sessions with the automatic resume, not one.
If the projection printed below exceeds what you are willing to spend, lower `MAX_STEPS` **before**
starting rather than killing the run later; the cosine schedule is defined over `MAX_STEPS`, so
changing it mid-run changes the learning-rate trajectory and makes the two halves incomparable.

Generation cost is estimated as `2 x DDIM_STEPS` UNet+control evaluations per image — the factor 2
is classifier-free guidance, which is the thing people forget when they budget for sampling.


In [ ]:
def bench_train_step(n_micro=None):
    n_micro = n_micro or cfg.GRAD_ACCUM
    unet.train(); controlnet.train()
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    it = iter(train_ld); ts = []
    optimizer.zero_grad(set_to_none=True)
    for i in range(n_micro):
        lat, cond, emb = next(it)
        torch.cuda.synchronize(); t0 = time.time()
        lat, cond, emb = lat.cuda(), cond.cuda(), emb.cuda()
        t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (lat.size(0),),
                          device="cuda")
        noise = torch.randn_like(lat)
        xt = noise_scheduler.add_noise(lat, noise, t)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            pred = cn_unet(xt, t, emb, cond)
        loss = F.mse_loss(pred.float(), noise.float())
        scaler.scale(loss / cfg.GRAD_ACCUM).backward()
        torch.cuda.synchronize(); ts.append(time.time() - t0)
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(LORA_PARAMS + CN_PARAMS, cfg.CLIP_NORM)
    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    peak = torch.cuda.max_memory_allocated() / 1e9
    return float(np.median(ts[1:] or ts)), peak

@torch.no_grad()
def bench_forward():
    unet.eval(); controlnet.eval()
    lat = torch.randn(1, 4, cfg.LATENT_SIZE, cfg.LATENT_SIZE, device="cuda")
    cond = torch.zeros(1, COND_CH, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda")
    emb = torch.zeros(1, 77, 768, device="cuda", dtype=torch.float16)
    t = torch.tensor([500], device="cuda")
    with torch.cuda.amp.autocast(enabled=cfg.AMP):
        for _ in range(3): cn_unet(lat, t, emb, cond)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(10): cn_unet(lat, t, emb, cond)
        torch.cuda.synchronize()
    return (time.time() - t0) / 10

fwd_s = bench_forward()
gen_s = fwd_s * cfg.DDIM_STEPS * 2          # x2 = classifier-free guidance

EFF = {"params_lora_M": n_lora / 1e6, "params_control_M": n_cn / 1e6,
       "params_frozen_unet_M": n_frozen / 1e6,
       "forward_s_per_eval": fwd_s, "gen_s_per_image_est": gen_s,
       "gen_min_for_EVAL_N": cfg.EVAL_N * gen_s / 60.0}

if TRAINABLE:
    micro_s, peak_gb = bench_train_step()
    step_s = micro_s * cfg.GRAD_ACCUM
    EFF.update({"micro_batch_s": micro_s, "optimizer_step_s": step_s,
                "peak_train_vram_gb": peak_gb,
                "projected_train_h": cfg.MAX_STEPS * step_s / 3600.0})
    optimizer, scheduler = build_optimizer()        # discard the benchmark's optimizer state
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.AMP)
    torch.cuda.empty_cache()
    print(f"MEASURED  micro-batch {micro_s:.2f} s (batch {cfg.BATCH_SIZE}) | "
          f"optimizer step {step_s:.1f} s (accum {cfg.GRAD_ACCUM}) | peak VRAM {peak_gb:.2f} GB "
          f"of {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"PROJECTED {cfg.MAX_STEPS} steps -> {EFF['projected_train_h']:.1f} h "
          f"({math.ceil(EFF['projected_train_h']/cfg.MAX_HOURS)} session(s) at "
          f"MAX_HOURS={cfg.MAX_HOURS})")
else:
    print("ablation trains nothing -- no training benchmark")

print(f"sampling  {fwd_s*1000:.0f} ms per UNet+control eval -> {gen_s:.1f} s per image at "
      f"DDIM {cfg.DDIM_STEPS} with CFG -> {EFF['gen_min_for_EVAL_N']:.0f} min for EVAL_N="
      f"{cfg.EVAL_N}")
print(json.dumps({k: round(v, 4) for k, v in EFF.items()}, indent=2))

## 15. Hugging Face persistence and resume

Same two-artifact layout as the segmentation notebooks and Stage 1, in this run's own repo:

- `best.pt` — LoRA weights + control-branch weights (fp16) + the run `cfg` + the architecture
  signature + metrics. This is what section 22's standalone sampler downloads.
- `last.pt` — full state incl. optimizer/scaler/scheduler, pushed every `PUSH_LAST_EVERY`
  **optimizer steps**. This is what a new session resumes from.

**Only what trains is saved.** The SD 1.5 UNet and the frozen VAE/text encoder are public
checkpoints and are re-downloaded from the Hub, so a checkpoint here is LoRA (~6 MB at r=16) plus
the control branch (~722 MB in fp16) — and for ablation C, where the branch is frozen, just the
LoRA. That is why `PUSH_LAST_EVERY` is 500 steps rather than the segmentation notebooks' 10 epochs:
with the optimizer state included, `last.pt` is roughly 1.5 GB and pushing it too often would spend
more of the session uploading than training. Set `PUSH_OPTIM_STATE=False` to halve that, at the cost
of restarting Adam's moments after a resume.

**The architecture signature is checked on load.** Loading a control branch trained under a
different conditioning-channel layout is the failure that produces six hours of plausible-looking
noise, so `check_arch` raises rather than warns.


In [ ]:
from huggingface_hub import HfApi, create_repo, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN"); print("secret fallback:", e)

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)
ABLATION_TABLE_REPO = f"{cfg.HF_USER}/{cfg.HF_PREFIX}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}-D"

def hf_push(local_path, name, msg, repo=None):
    try:
        api.upload_file(path_or_fileobj=local_path, path_in_repo=name,
                        repo_id=repo or HF_REPO, token=HF_TOKEN, commit_message=msg)
    except Exception as e:
        print("  HF push failed (continuing):", type(e).__name__, e)

def hf_try_download(name, repo=None):
    try: return hf_hub_download(repo or HF_REPO, name, token=HF_TOKEN)
    except Exception: return None

def check_arch(st):
    got = st.get("arch")
    if got != ARCH_SIG:
        raise RuntimeError("architecture signature mismatch, refusing to load\n"
                           f"  checkpoint: {got}\n  this run  : {ARCH_SIG}")

def lora_state():
    return {k: v.detach().half().cpu() for k, v in unet.state_dict().items() if "lora" in k}

def control_state():
    if not cfg.TRAIN_CONTROLNET: return None
    return {k: v.detach().half().cpu() for k, v in controlnet.state_dict().items()}

def load_weights(st):
    check_arch(st)
    if st.get("lora_state"):
        missing, unexpected = unet.load_state_dict(
            {k: v.float() for k, v in st["lora_state"].items()}, strict=False)
        assert not unexpected, f"unexpected LoRA keys: {list(unexpected)[:4]}"
    if st.get("control_state"):
        controlnet.load_state_dict({k: v.float() for k, v in st["control_state"].items()})

def push_model_card(final, step):
    lay = final.get("layout", {})
    L = ["---",
         "tags: [controlnet, lora, stable-diffusion, remote-sensing, geonusaf]",
         "library_name: diffusers", "---", "",
         f"# GeoNUSAF Stage 2 - mask-conditioned satellite image generator "
         f"({cfg.SPLIT_MODE} fold {cfg.FOLD}, ablation {WEIGHTS_ABLATION})",
         "",
         "Paints a Kathmandu Valley satellite image onto a 7-class land-use mask. Trained on the",
         "fold-0 TRAIN tiles only; validation tiles never enter a gradient.", "",
         "| field | value |", "|---|---|",
         f"| base | {cfg.SD_MODEL_ID} (frozen) |",
         f"| control branch | {cfg.CONTROLNET_ID}, warm-started, "
         f"{'fine-tuned' if cfg.TRAIN_CONTROLNET else 'frozen'} |",
         f"| condition | {cfg.COND_MODE}, {COND_CH} channels, init {cfg.COND_CONV_INIT} |",
         f"| LoRA | {'r%d alpha %d on to_q/to_k/to_v/to_out.0' % (cfg.LORA_RANK, cfg.LORA_ALPHA) if cfg.USE_LORA else 'none'} |",
         f"| lr | LoRA {cfg.LR_LORA}, control {cfg.LR_CONTROL} |",
         f"| resolution | {cfg.IMG_SIZE}, GSD 0.586 m/px |",
         f"| sampling | DDIM {cfg.DDIM_STEPS} steps, guidance {cfg.GUIDANCE} |",
         f"| split | {cfg.SPLIT_MODE} fold {cfg.FOLD}, train_sha1 {split_sha1[:12]} |",
         f"| seed | {cfg.SEED} | ",
         f"| steps | {step} |",
         f"| layout mIoU (gen) | {lay.get('miou_gen', float('nan')):.4f} |",
         f"| layout mIoU (real ceiling) | {lay.get('miou_real', float('nan')):.4f} |",
         f"| layout ratio | {lay.get('layout_ratio', float('nan')):.3f} |",
         f"| KID | {final.get('KID_mean', float('nan')):.5f} +/- {final.get('KID_std', float('nan')):.5f} |",
         f"| FID | {final.get('FID', float('nan')):.2f} |", "",
         "## Input contract",
         f"Single-channel class-index PNG, {cfg.IMG_SIZE}x{cfg.IMG_SIZE}, values `{{0..5}}` with "
         "`255` (or `6`) for ignore. Same format whether the mask is real or produced by Stage 1.",
         "", "## Output", "`genimage{i}.png` + `genmask{i}.png`, paired, ready for segmentation training."]
    p = f"{cfg.OUT_DIR}/README.md"; open(p, "w").write("\n".join(L))
    hf_push(p, "README.md", f"card step {step}")

print("HF ready:", HF_REPO, "| ablation table repo:", ABLATION_TABLE_REPO)

## 16. Weights & Biases

Matches the logging on the segmentation baselines and Stage 1. `resume="allow"` with a deterministic
run id means a resumed session continues the same run rather than starting a second one.

Two series here that the other notebooks do not have:
- **`val_loss`** — fixed-noise eps-MSE on the held-out val tiles. For a diffusion finetune the train
  loss is nearly flat and uninformative after the first few hundred steps, which is normal and is
  *not* a stalled run. The val curve is what model selection uses.
- **validation image grids** every `SAMPLE_EVERY` steps: real image, its mask, and the image
  generated from that mask. For a generator the grid *is* the training curve. What to watch for, in
  order: first the colours become plausible (LoRA working), then the boundaries start to line up
  with the mask (control branch working). If step 1500 still shows plausible Kathmandu texture with
  no relation to the mask, the control branch is not learning and section 19 will confirm it.


In [ ]:
import wandb
try:
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    WANDB = True
except Exception as e:
    print("wandb disabled:", e); WANDB = False

if WANDB:
    wandb.init(project=cfg.WANDB_PROJECT, name=f"{cfg.HF_PREFIX}-{tag}", id=f"{cfg.HF_PREFIX}-{tag}",
               resume="allow", config=asdict(cfg))
    wandb.config.update({"arch_sig": ARCH_SIG, "n_lora": n_lora, "n_control": n_cn,
                         "cond_channels": COND_CH, "train_tiles": len(tr_ids),
                         "train_pairs": len(train_ds), "split_sha1": split_sha1},
                        allow_val_change=True)
print("wandb:", WANDB)

## 17. Train step, validation monitor, and the DDIM sampler

### Training
One optimizer step is `GRAD_ACCUM` micro-batches of: draw `t ~ U{0..999}`, draw `eps ~ N(0,I)`, form
the noisy latent, predict `eps` through control branch -> frozen UNet, take the MSE. The loss is
computed **in fp32 outside autocast** for the same reason the segmentation notebooks do it — fp16
MSE on a near-zero residual loses precision exactly where it matters, late in training.

`SNR_GAMMA=5.0` applies **min-SNR-gamma** weighting (Hang et al., 2023): the per-sample loss is
scaled by `min(SNR_t, gamma)/SNR_t`, which caps the weight the low-noise timesteps would otherwise
carry. Without it, a short finetune spends most of its gradient on timesteps that are nearly clean
already, and the layout — which is decided at high noise — trains slowest. Set it to `None` for
plain MSE if you want that as an ablation row.

`scaler.unscale_` runs before `clip_grad_norm_`, and both trainable groups are clipped together so
the ratio between the LoRA and control-branch updates is preserved. Both are copied from the part-1
notebooks deliberately.

### Validation monitor
`validate()` computes the same eps-MSE on the **fold-0 val tiles** with noise and timesteps drawn
from a fixed seed, so the number is comparable step to step rather than being dominated by which
timesteps happened to come up. No gradient is taken and no val tile ever reaches the optimizer.

### Sampler
DDIM, `DDIM_STEPS` steps, classifier-free guidance at `GUIDANCE`, one scheduler built per batch so a
partially-consumed timestep list can never leak between calls. The seed is derived from `cfg.SEED`
plus the batch offset, so sampling the same seed twice gives the same images and the grids in the
training log are a genuine like-for-like progression.

The canny branch is loaded **lazily and in fp16**, only if `USE_CANNY`, and is never in the training
graph — which is what makes ablation E a re-sample of D rather than a second training run. Its
condition is the exact class-boundary map from section 9, so it reinforces the same edges the seg
branch is already being asked to respect rather than importing structure from anywhere else.


In [ ]:
DDIM_CFG = DDIMScheduler.from_pretrained(cfg.SD_MODEL_ID, subfolder="scheduler").config
_CANNY_CN = None

def get_canny_cn():
    global _CANNY_CN
    if _CANNY_CN is None:
        _CANNY_CN = ControlNetModel.from_pretrained(
            cfg.CANNY_ID, torch_dtype=torch.float16).cuda().eval()
        _CANNY_CN.requires_grad_(False)
        print("  canny branch loaded (fp16, frozen, sampling only)")
    return _CANNY_CN

def snr_weight(t):
    acp = noise_scheduler.alphas_cumprod.to(t.device)[t]
    snr = acp / (1.0 - acp)
    return torch.minimum(snr, torch.full_like(snr, float(cfg.SNR_GAMMA))) / snr

def diffusion_loss(pred, noise, t):
    per = ((pred.float() - noise.float()) ** 2).mean(dim=(1, 2, 3))
    if cfg.SNR_GAMMA is not None: per = per * snr_weight(t)
    return per.mean()

def infinite(loader, ds):
    ep = 0
    while True:
        ds.epoch = ep
        for b in loader: yield b
        ep += 1

def train_step(it):
    unet.train(); controlnet.train()
    tot = 0.0
    optimizer.zero_grad(set_to_none=True)
    for _ in range(cfg.GRAD_ACCUM):
        lat, cond, emb = next(it)
        lat, cond, emb = lat.cuda(non_blocking=True), cond.cuda(non_blocking=True), emb.cuda(non_blocking=True)
        t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (lat.size(0),), device="cuda")
        noise = torch.randn_like(lat)
        xt = noise_scheduler.add_noise(lat, noise, t)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            pred = cn_unet(xt, t, emb, cond)
        loss = diffusion_loss(pred, noise, t)                    # fp32, outside autocast
        scaler.scale(loss / cfg.GRAD_ACCUM).backward()
        tot += float(loss)
    scaler.unscale_(optimizer)                                   # unscale BEFORE clipping
    gn = torch.nn.utils.clip_grad_norm_(LORA_PARAMS + CN_PARAMS, cfg.CLIP_NORM)
    prev_scale = scaler.get_scale()
    scaler.step(optimizer); scaler.update()
    optimizer.zero_grad(set_to_none=True); scheduler.step()
    return tot / cfg.GRAD_ACCUM, float(gn), scaler.get_scale() >= prev_scale

@torch.no_grad()
def validate():
    unet.eval(); controlnet.eval()
    tot, n = 0.0, 0
    for bi, (lat, cond, emb) in enumerate(val_ld):
        if bi >= cfg.VAL_BATCHES: break
        lat, cond, emb = lat.cuda(), cond.cuda(), emb.cuda()
        gg = torch.Generator(); gg.manual_seed(cfg.VAL_NOISE_SEED + bi)    # fixed noise/timesteps
        noise = torch.randn(lat.shape, generator=gg).cuda()
        t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (lat.size(0),),
                          generator=gg).cuda()
        xt = noise_scheduler.add_noise(lat, noise, t)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            pred = cn_unet(xt, t, emb, cond)
        tot += float(diffusion_loss(pred, noise, t)) * lat.size(0); n += lat.size(0)
    return tot / max(n, 1)

def embed_cached(caps):
    miss = sorted({c for c in caps if c not in TEXTEMB})
    if miss:
        for c, v in zip(miss, embed_captions(miss).cpu()): TEXTEMB[c] = v
    return torch.stack([TEXTEMB[c] for c in caps]).cuda()

@torch.no_grad()
def vae_decode(lat):
    img = vae.decode((lat / SCALING).float()).sample
    return (img.float() / 2 + 0.5).clamp(0, 1)

@torch.no_grad()
def generate_images(labels, seed=0, steps=None, guidance=None, cond_scale=None,
                    use_canny=None, batch=None, progress=False):
    # labels: list/array of (H, W) class-index masks. Returns (N, H, W, 3) uint8 RGB.
    steps      = steps or cfg.DDIM_STEPS
    guidance   = cfg.GUIDANCE if guidance is None else guidance
    cond_scale = cfg.COND_SCALE if cond_scale is None else cond_scale
    use_canny  = cfg.USE_CANNY if use_canny is None else use_canny
    batch      = batch or cfg.EVAL_BATCH
    do_cfg     = guidance > 1.0
    unet.eval(); controlnet.eval()
    out = []
    for s in range(0, len(labels), batch):
        chunk = list(labels[s:s + batch]); b = len(chunk)
        cond = torch.from_numpy(np.stack([cond_from_index(l) for l in chunk])).cuda()
        emb  = embed_cached([caption_from_index(l) for l in chunk])
        neg  = embed_cached([""] * b)
        e_all = torch.cat([neg, emb]) if do_cfg else emb
        c_all = torch.cat([cond, cond]) if do_cfg else cond
        extra = []
        if use_canny:
            cc = torch.from_numpy(np.stack([canny_from_index(l) for l in chunk])).cuda()
            extra = [(get_canny_cn(), torch.cat([cc, cc]) if do_cfg else cc, cfg.CANNY_SCALE)]
        sch = DDIMScheduler.from_config(DDIM_CFG); sch.set_timesteps(steps, device="cuda")
        gg = torch.Generator(); gg.manual_seed(cfg.SEED + seed + s)
        lat = (torch.randn(b, 4, cfg.LATENT_SIZE, cfg.LATENT_SIZE, generator=gg).cuda()
               * sch.init_noise_sigma)
        for t in sch.timesteps:
            x = torch.cat([lat] * 2) if do_cfg else lat
            x = sch.scale_model_input(x, t)
            with torch.cuda.amp.autocast(enabled=cfg.AMP):
                eps = cn_unet(x, t, e_all, c_all, cond_scale, extra)
            eps = eps.float()
            if do_cfg:
                u, c_ = eps.chunk(2); eps = u + guidance * (c_ - u)
            lat = sch.step(eps, t, lat).prev_sample
        img = vae_decode(lat).cpu().numpy().transpose(0, 2, 3, 1)
        out.append((img * 255).round().astype(np.uint8))
        if progress: print(f"  generated {min(s+b, len(labels))}/{len(labels)}", flush=True)
    return np.concatenate(out)

def val_label(i):
    r = va_recs[i]
    return RAM[r[2]][1] if cfg.PRELOAD_RAM else cv2.imread(str(CACHE / "labels" / r[3]),
                                                           cv2.IMREAD_GRAYSCALE)

_t0 = time.time()
_smoke = generate_images([val_label(0)], seed=0, steps=8)
print(f"smoke sample: {_smoke.shape} in {time.time()-_t0:.1f}s at 8 DDIM steps "
      f"(untrained control branch -> expect texture, not layout)")

## 18. Main loop with automatic resume

On start this pulls `last.pt` from the repo if it exists and continues from that step, with the
architecture signature checked first. So the answer to "what if it exceeds 12 hours" is the same as
in the other notebooks: it stops, you start a new session, run the notebook top to bottom unchanged,
and it picks up where it left off. `MAX_HOURS` makes the stop graceful rather than letting Kaggle
kill the kernel mid-step.

**Expected cost** is whatever section 14 measured — read that number, not this paragraph. As a rough
anchor: ~12-20 s per optimizer step on one T4, so `MAX_STEPS=3000` is 10-16 h, i.e. two sessions.

**What the curves should look like.** The eps-MSE drops for a few hundred steps and then sits almost
flat. That is normal for diffusion and is not a stalled run — the loss averages over all 1000
timesteps and is dominated by the high-noise ones, which are easy. **Judge progress from the sample
grids, not the loss.** The one curve worth watching is `val_loss`: if it climbs while `train_loss`
falls, the model is memorizing 270 tiles, and the layout numbers in section 19 will not save it.

**No early stopping.** There is no metric that saturates cleanly here, so the run is meant to reach
the end of the cosine. `best.pt` tracks the lowest val eps-MSE purely so that a session that dies
late still leaves a defensible checkpoint behind.

Ablations A and B train nothing, so this cell short-circuits and section 19 evaluates the
off-the-shelf pipeline directly — which is exactly what those rows are for.


In [ ]:
history, step, best_val, best_step = [], 0, float("inf"), -1

if not TRAINABLE:
    print(f"ablation {cfg.ABLATION} trains nothing (no LoRA, frozen control branch) "
          "-- skipping straight to evaluation")
else:
    ck = hf_try_download("last.pt")
    if ck:
        st = torch.load(ck, map_location="cuda")
        load_weights(st)
        if st.get("optim_state"): optimizer.load_state_dict(st["optim_state"])
        if st.get("sched_state"): scheduler.load_state_dict(st["sched_state"])
        if st.get("scaler_state"): scaler.load_state_dict(st["scaler_state"])
        step = st["step"] + 1; best_val = st["best_val"]; best_step = st["best_step"]
        history = st["history"]
        print(f"RESUMED from step {st['step']} (best val {best_val:.5f} @ s{best_step})")
    else:
        print("fresh start")

    torch.cuda.reset_peak_memory_stats()
    t_start = time.time()
    data_it = infinite(train_ld, train_ds)

    while step < cfg.MAX_STEPS:
        ts = time.time()
        tr_loss, gn, took = train_step(data_it)
        row = {"step": step, "train_loss": tr_loss, "grad_norm": gn,
               "lr_lora": scheduler.get_last_lr()[0],
               "lr_control": scheduler.get_last_lr()[-1],
               "step_sec": time.time() - ts,
               "peak_vram_gb": torch.cuda.max_memory_allocated() / 1e9,
               "hours": (time.time() - t_start) / 3600}

        if cfg.VAL_MONITOR and (step % cfg.VAL_EVERY == 0 or step == cfg.MAX_STEPS - 1):
            row["val_loss"] = validate()
            score = row["val_loss"]
            if score == score and score < best_val:                 # NaN-safe
                best_val, best_step = score, step
                torch.save({"lora_state": lora_state(), "control_state": control_state(),
                            "arch": ARCH_SIG, "cfg": asdict(cfg), "step": step,
                            "val_loss": best_val}, f"{cfg.OUT_DIR}/best.pt")
        history.append(row)
        if WANDB: wandb.log(row, step=step)

        if step % cfg.LOG_EVERY == 0 or step == cfg.MAX_STEPS - 1:
            print(f"[s{step}] train {tr_loss:.5f} | val {row.get('val_loss', float('nan')):.5f} | "
                  f"gnorm {gn:.2f} | lr {row['lr_lora']:.2e}/{row['lr_control']:.2e} | "
                  f"{row['step_sec']:.1f}s | vram {row['peak_vram_gb']:.1f}GB | "
                  f"{row['hours']:.2f}h", flush=True)

        if (step + 1) % cfg.SAMPLE_EVERY == 0:
            idx = list(range(0, len(va_recs), max(1, len(va_recs) // cfg.N_SAMPLE_GRID)))[:cfg.N_SAMPLE_GRID]
            labs = [val_label(i) for i in idx]
            gens = generate_images(labs, seed=step)
            reals = [cv2.cvtColor(cv2.imread(str(CACHE / "images" / va_recs[i][2]),
                                             cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB) for i in idx]
            fig = triptych(reals, labs, gens, n=len(idx), title=f"step {step}")
            pg = f"{cfg.OUT_DIR}/samples_s{step}.png"
            plt.savefig(pg, dpi=110, bbox_inches="tight"); plt.show()
            if WANDB: wandb.log({"samples": wandb.Image(pg)}, step=step)

        last_step = (step == cfg.MAX_STEPS - 1)
        out_of_time = (time.time() - t_start) / 3600 > cfg.MAX_HOURS
        if (step % cfg.PUSH_LAST_EVERY == 0 and step > 0) or last_step or out_of_time:
            p = f"{cfg.OUT_DIR}/last.pt"
            torch.save({"lora_state": lora_state(), "control_state": control_state(),
                        "optim_state": optimizer.state_dict() if cfg.PUSH_OPTIM_STATE else None,
                        "sched_state": scheduler.state_dict(),
                        "scaler_state": scaler.state_dict(),
                        "arch": ARCH_SIG, "step": step, "best_val": best_val,
                        "best_step": best_step, "history": history}, p)
            hf_push(p, "last.pt", f"state s{step}")
            if os.path.exists(f"{cfg.OUT_DIR}/best.pt"):
                hf_push(f"{cfg.OUT_DIR}/best.pt", "best.pt", f"best val {best_val:.5f} @ s{best_step}")

        if out_of_time:
            print(f"TIME BUDGET reached at step {step} ({cfg.MAX_HOURS} h) -- state pushed, "
                  "re-run the notebook in a new session to continue"); break
        step += 1

    wall_h = (time.time() - t_start) / 3600
    print(f"training done | best val {best_val:.5f} @ step {best_step} | this session {wall_h:.2f} h")

## 19. Final evaluation of the best checkpoint

Reloads `best.pt`, generates `EVAL_N` images from **held-out fold-0 validation masks**, and runs the
full suite. Only the `orig` variant of each val tile is used — including the hf/vf copies would put
three near-identical scenes into a KID estimate that assumes independent samples and would flatter
the result.

### Reading the layout numbers, in this order
1. `layout_ratio = miou_gen / miou_real`. **This is the number.** `miou_real` is the same frozen
   SegFormer-B0 scored on the *real* val images against the same masks, so it is the ceiling this
   measurement can reach — around 0.29 on fold 0. A ratio of 0.60 or better means the generated
   layout is close to as segmentable as the real thing.
2. `miou_shuffled`. The null: the same generated images scored against *other* scenes' masks. It
   costs nothing and it is what rules out "the segmenter just predicts Residential everywhere". If
   `miou_gen` is not comfortably above it, the generator is not following the mask, whatever the
   absolute number says.
3. The absolute floor `LAYOUT_MIOU_FLOOR=0.30`, kept because it was specified — but note in the
   paper that it sits *above* the real-image ceiling for this fold, so it should be read as a
   stretch target rather than a pass condition.

### KID and FID
KID is the headline and both N values are printed with it. FID is reported for completeness with
its N attached; at a few hundred scenes its bias is large enough that ranking two generators by it
is not defensible.

### clDice
Road and River, between the conditioning mask and the re-segmented output. A generator can score
respectably on mIoU while turning a 9 px road into a dotted smear; clDice is what catches that,
and it is the same implementation the part-1 notebooks use, so the numbers are comparable.


In [ ]:
_best_path = f"{cfg.OUT_DIR}/best.pt"
if TRAINABLE and not os.path.exists(_best_path):
    _dl = hf_try_download("best.pt")          # resumed session that never wrote one locally
    if _dl: _best_path = _dl

if TRAINABLE and os.path.exists(_best_path):
    _ck = torch.load(_best_path, map_location="cuda")
    load_weights(_ck)
    print(f"loaded best.pt from step {_ck['step']} (val {_ck['val_loss']:.5f})")
    best_step_used, best_val_used = int(_ck["step"]), float(_ck["val_loss"])
else:
    print("evaluating the current (off-the-shelf) weights -- no trained checkpoint for this row")
    best_step_used, best_val_used = -1, float("nan")

ORIG_VAL = [i for i, r in enumerate(va_recs) if r[1] == "orig"]
EVAL_IDX = ORIG_VAL[:min(cfg.EVAL_N, len(ORIG_VAL))]
EVAL_LAB = [val_label(i) for i in EVAL_IDX]

def real_rgb01(idxs):
    ims = [cv2.cvtColor(cv2.imread(str(CACHE / "images" / va_recs[i][2]), cv2.IMREAD_COLOR),
                        cv2.COLOR_BGR2RGB) for i in idxs]
    return torch.from_numpy(np.stack(ims).transpose(0, 3, 1, 2).astype(np.float32) / 255.0)

REAL01_ALL  = real_rgb01(ORIG_VAL)            # the real reference set for KID and FID
REAL01_EVAL = REAL01_ALL[:len(EVAL_IDX)]      # the matched subset, for the mIoU ceiling
F_REAL = inception_feats(REAL01_ALL)

def evaluate_generated(labels, use_canny=None, seed=101, name=""):
    t0 = time.time()
    gens = generate_images(labels, seed=seed, use_canny=use_canny, progress=True)
    gen_min = (time.time() - t0) / 60
    g01 = torch.from_numpy(gens.transpose(0, 3, 1, 2).astype(np.float32) / 255.0)

    pred_gen  = segment_rgb01(segmodel, g01).numpy()
    pred_real = segment_rgb01(segmodel, REAL01_ALL[:len(labels)]).numpy()
    shuf = list(np.roll(np.arange(len(labels)), 1))

    m_gen  = layout_scores(pred_gen, labels, cfg.CLASS_NAMES)
    m_shuf = layout_scores(pred_gen, [labels[i] for i in shuf], cfg.CLASS_NAMES)
    m_real = layout_scores(pred_real, labels, cfg.CLASS_NAMES)

    f_gen = inception_feats(g01)
    fid = fid_from_feats(F_REAL, f_gen)
    kid_m, kid_s, kid_sub, kid_n = kid_from_feats(F_REAL, f_gen, seed=cfg.SEED)
    ratio = m_gen["mIoU"] / max(m_real["mIoU"], 1e-9)

    res = {"name": name or cfg.ABLATION,
           "layout": {"miou_gen": m_gen["mIoU"], "miou_shuffled": m_shuf["mIoU"],
                      "miou_real": m_real["mIoU"], "layout_ratio": ratio,
                      "lift_over_null": m_gen["mIoU"] - m_shuf["mIoU"],
                      "mF1_gen": m_gen["mF1"], "OA_gen": m_gen["OA"], "kappa_gen": m_gen["kappa"],
                      "per_class_gen": m_gen["per_class"]},
           "clDice": {"Road_gen": m_gen["clDice_Road"], "River_gen": m_gen["clDice_River"],
                      "Road_real": m_real["clDice_Road"], "River_real": m_real["clDice_River"]},
           "KID_mean": kid_m, "KID_std": kid_s, "KID_subset_size": kid_sub, "KID_subsets": kid_n,
           "FID": fid, "N_real": int(len(REAL01_ALL)), "N_gen": int(len(labels)),
           "generation_minutes": gen_min,
           "verdict_ratio": "PASS" if ratio >= cfg.LAYOUT_RATIO_FLOOR else "FAIL",
           "verdict_null": "PASS" if m_gen["mIoU"] > m_shuf["mIoU"] * 1.5 else "FAIL",
           "verdict_absolute": "PASS" if m_gen["mIoU"] >= cfg.LAYOUT_MIOU_FLOOR else "FAIL"}
    return res, gens

RES, GEN = evaluate_generated(EVAL_LAB, name=cfg.ABLATION)
np.save(f"{cfg.OUT_DIR}/gen_{tag}.npy", GEN[:16])

lay = RES["layout"]
print(f"\n--- layout fidelity (N={RES['N_gen']} held-out val masks) ---")
print(f"  generated vs its own mask   mIoU {lay['miou_gen']:.4f}   <- the measurement")
print(f"  generated vs a WRONG mask   mIoU {lay['miou_shuffled']:.4f}   <- the null")
print(f"  REAL image vs the same mask mIoU {lay['miou_real']:.4f}   <- the ceiling")
print(f"  ratio gen/real {lay['layout_ratio']:.3f} ({RES['verdict_ratio']} at "
      f">={cfg.LAYOUT_RATIO_FLOOR}) | lift over null {lay['lift_over_null']:+.4f} "
      f"({RES['verdict_null']}) | absolute floor {RES['verdict_absolute']}")

pc = pd.DataFrame(lay["per_class_gen"]).T[["IoU", "F1", "producer_acc", "user_acc"]]
print("\n--- per-class on generated images ---")
print(pc.round(4).to_string())

print(f"\n--- realism ---")
print(f"  KID  {RES['KID_mean']:.5f} +/- {RES['KID_std']:.5f}   "
      f"(N_real={RES['N_real']}, N_gen={RES['N_gen']}, "
      f"subset={RES['KID_subset_size']} x {RES['KID_subsets']} subsets)   <- headline")
print(f"  FID  {RES['FID']:.2f}   (N_real={RES['N_real']}, N_gen={RES['N_gen']}; "
      "biased at this N, secondary)")
print(f"\n--- clDice on thin structures (mask vs re-segmented output) ---")
for k in ("Road", "River"):
    print(f"  {k:<6} generated {RES['clDice'][k+'_gen']:.4f} | real {RES['clDice'][k+'_real']:.4f}")

final = {"run": {"model": "SD15-LoRA-ControlNet", "stage": 2, "ablation": cfg.ABLATION,
                 "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "seed": cfg.SEED,
                 "train_sha1": split_sha1, "arch": ARCH_SIG},
         "best_step": best_step_used, "best_val_eps_mse": best_val_used,
         "steps_run": int(len(history)), **RES}
final.update(EFF)
print("\nverdicts:", {k: v for k, v in RES.items() if k.startswith("verdict")})

## 20. Persist results

Writes `results_{mode}_fold{k}_{ablation}.json` (full), a flat `.csv` row for the cross-stage
comparison table, and `history_{...}.csv` for the curves — then pushes all three plus the model card
to the run's HF repo so they survive the session. `fold0_split.json` goes up too, so that Stage 1's
copy and this one can be diffed directly rather than argued about.


In [ ]:
jp = f"{cfg.OUT_DIR}/results_{tag}.json"
json.dump(final, open(jp, "w"), indent=2, default=str)

flat = {"model": "SD15-LoRA-ControlNet", "stage": 2, "ablation": cfg.ABLATION,
        "cond_mode": cfg.COND_MODE, "cond_channels": COND_CH, "lora": cfg.USE_LORA,
        "control_trained": cfg.TRAIN_CONTROLNET, "canny": cfg.USE_CANNY,
        "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "seed": cfg.SEED,
        "best_step": final["best_step"], "val_eps_mse": final["best_val_eps_mse"],
        "miou_gen": lay["miou_gen"], "miou_shuffled": lay["miou_shuffled"],
        "miou_real": lay["miou_real"], "layout_ratio": lay["layout_ratio"],
        "mF1_gen": lay["mF1_gen"], "OA_gen": lay["OA_gen"], "kappa_gen": lay["kappa_gen"],
        "clDice_Road": RES["clDice"]["Road_gen"], "clDice_River": RES["clDice"]["River_gen"],
        "KID_mean": RES["KID_mean"], "KID_std": RES["KID_std"], "FID": RES["FID"],
        "N_real": RES["N_real"], "N_gen": RES["N_gen"],
        "verdict_ratio": RES["verdict_ratio"], "verdict_null": RES["verdict_null"],
        "params_lora_M": EFF["params_lora_M"], "params_control_M": EFF["params_control_M"],
        "gen_s_per_image": EFF["gen_s_per_image_est"],
        "peak_vram_gb": EFF.get("peak_train_vram_gb", float("nan")),
        "train_sha1": split_sha1}
for k, v in lay["per_class_gen"].items(): flat[f"IoU_{k}"] = v["IoU"]

cp = f"{cfg.OUT_DIR}/results_{tag}.csv"
with open(cp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(flat)); w.writeheader(); w.writerow(flat)

paths = [(jp, f"results_{tag}.json"), (cp, f"results_{tag}.csv"),
         (f"{cfg.OUT_DIR}/fold0_split.json", "fold0_split.json")]
if history:
    hp = f"{cfg.OUT_DIR}/history_{tag}.csv"
    with open(hp, "w", newline="") as f:
        keys = sorted({k for r in history for k in r})
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(history)
    paths.append((hp, f"history_{tag}.csv"))

for p, n in paths: hf_push(p, n, "results")
push_model_card(final, final["best_step"])

if WANDB:
    wandb.log({f"final/{k}": v for k, v in flat.items() if isinstance(v, (int, float))})
print("results pushed to", HF_REPO)

## 21. Sampling interface — mask PNGs in, paired PNGs out

**This is the interface contract. Do not change anything in this cell without telling Stage 1.**

Input: single-channel class-index PNGs at 512x512, values `{0..5}` with `255` for ignore. Code `6`
is accepted as an alias for ignore so that Stage 1's internal coding also drops in unchanged. Point
`SAMPLE_IN_DIR` at Stage 1's `genmask*.png` directory, or leave it `None` to sample from the fold-0
validation masks — **the format is identical either way**, which is the whole point of the contract.

Output: `genimage{i}.png` (BGR, as `cv2.imwrite` writes it, matching how the segmentation notebooks
read images) alongside a byte-copy of the conditioning mask as `genmask{i}.png`. The pair drops
straight into part-1 segmentation training with no adapter.

The asserts re-read a written file and check it against the contract: 3-channel image, single-channel
mask, both `uint8` at 512x512, mask values in `{0..5, 255}`. Deliberately paranoid — a silent
channel-order flip here would not surface until the stage-4 augmentation experiment produced an
inexplicably worse model a week later.


In [ ]:
def read_index_png(p):
    a = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    assert a is not None and a.ndim == 2, f"{p} is not a single-channel PNG"
    if a.shape != (cfg.IMG_SIZE, cfg.IMG_SIZE):
        a = cv2.resize(a, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    vals = set(np.unique(a).tolist())
    assert vals <= set(range(cfg.N_CLASSES)) | {cfg.IGNORE_CODE, cfg.IGNORE_INDEX}, \
        f"{p} has illegal values {sorted(vals)}"
    return np.where(a == cfg.IGNORE_CODE, cfg.IGNORE_INDEX, a).astype(np.uint8)

if cfg.SAMPLE_IN_DIR:
    src = sorted(Path(cfg.SAMPLE_IN_DIR).glob("*.png"))[:cfg.N_SAMPLE_OUT]
    SAMPLE_LABS = [read_index_png(p) for p in src]
    SOURCE = f"{cfg.SAMPLE_IN_DIR} ({len(src)} files)"
else:
    idxs = ORIG_VAL[:cfg.N_SAMPLE_OUT]
    SAMPLE_LABS = [val_label(i) for i in idxs]
    SOURCE = f"fold-0 validation masks ({len(idxs)} tiles)"
print(f"conditioning source: {SOURCE}")

for f in Path(cfg.SAMPLE_OUT_DIR).glob("gen*.png"): f.unlink()      # never mix two runs
t0 = time.time()
OUT_IMGS = generate_images(SAMPLE_LABS, seed=cfg.SAMPLE_SEED, progress=True)
for i, (im, lb) in enumerate(zip(OUT_IMGS, SAMPLE_LABS)):
    cv2.imwrite(f"{cfg.SAMPLE_OUT_DIR}/genimage{i}.png",
                cv2.cvtColor(im, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_PNG_COMPRESSION, 3])
    cv2.imwrite(f"{cfg.SAMPLE_OUT_DIR}/genmask{i}.png", lb, [cv2.IMWRITE_PNG_COMPRESSION, 3])
print(f"wrote {len(OUT_IMGS)} pairs to {cfg.SAMPLE_OUT_DIR} in {(time.time()-t0)/60:.1f} min")

# ---- contract check ----
ci = cv2.imread(f"{cfg.SAMPLE_OUT_DIR}/genimage0.png", cv2.IMREAD_UNCHANGED)
cm_ = cv2.imread(f"{cfg.SAMPLE_OUT_DIR}/genmask0.png", cv2.IMREAD_UNCHANGED)
assert ci.shape == (cfg.IMG_SIZE, cfg.IMG_SIZE, 3) and ci.dtype == np.uint8, ci.shape
assert cm_.ndim == 2 and cm_.shape == (cfg.IMG_SIZE, cfg.IMG_SIZE) and cm_.dtype == np.uint8, cm_.shape
seen = set()
for p in sorted(Path(cfg.SAMPLE_OUT_DIR).glob("genmask*.png"))[::7]:
    seen |= set(np.unique(cv2.imread(str(p), cv2.IMREAD_UNCHANGED)).tolist())
assert seen <= set(range(cfg.N_CLASSES)) | {cfg.IGNORE_INDEX}, f"illegal mask values: {sorted(seen)}"
print(f"contract OK | image {ci.shape} {ci.dtype} | mask {cm_.shape} {cm_.dtype} | "
      f"mask values {sorted(seen)}")

manifest = {"produced_by": RUN_NAME, "ablation": cfg.ABLATION, "hf_repo": HF_REPO,
            "arch": ARCH_SIG, "best_step": final["best_step"], "n_pairs": len(OUT_IMGS),
            "size": cfg.IMG_SIZE, "conditioning_source": SOURCE,
            "naming": "genimage{i}.png + genmask{i}.png",
            "image_channel_order": "BGR on disk (cv2), as the part-1 loaders read it",
            "mask_values": "0..5 semantic, 255 ignore",
            "sampler": f"DDIM {cfg.DDIM_STEPS} steps, guidance {cfg.GUIDANCE}, "
                       f"cond_scale {cfg.COND_SCALE}, canny {cfg.USE_CANNY}",
            "class_names": list(cfg.CLASS_NAMES), "train_sha1": split_sha1,
            "layout_ratio": lay["layout_ratio"], "KID_mean": RES["KID_mean"]}
mp = f"{cfg.OUT_DIR}/fake_pairs_manifest.json"
json.dump(manifest, open(mp, "w"), indent=2)

TAR = f"{cfg.OUT_DIR}/fake_pairs_{tag}.tar.gz"
with tarfile.open(TAR, "w:gz") as tf:
    tf.add(cfg.SAMPLE_OUT_DIR, arcname="fake_pairs")
hf_push(mp, "fake_pairs_manifest.json", "hand-off manifest")
hf_push(TAR, os.path.basename(TAR), "generated image-mask pairs")
print(f"archive: {TAR} ({os.path.getsize(TAR)/1e6:.1f} MB) -> pushed to {HF_REPO}")
print("\nStage 4 input: pull the archive above, or Save Version and attach "
      f"{cfg.SAMPLE_OUT_DIR} as a Kaggle dataset.")

## 22. Standalone generation from Hugging Face

Self-contained, exactly like the inference cell in the segmentation notebooks: point `GEN_REPO` at
this run's repo, give it a directory of class-index masks, get paired PNGs back. Nothing above this
cell is required — the base SD 1.5 and the ControlNet topology come from the Hub, and `best.pt`
supplies the LoRA and control-branch weights plus the `cfg` that says which condition format they
were trained for.

Paste it into a fresh notebook as-is when stage 4 wants more pairs than the hand-off set contains,
or a second set at a different seed for a variance estimate.


In [ ]:
def load_generator_from_hf(repo_id, token=None, device="cuda"):
    import torch, numpy as np
    from diffusers import UNet2DConditionModel, ControlNetModel, AutoencoderKL, DDIMScheduler
    from transformers import CLIPTextModel, CLIPTokenizer
    from huggingface_hub import hf_hub_download
    ck = torch.load(hf_hub_download(repo_id, "best.pt", token=token), map_location=device)
    c = ck["cfg"]
    cond_ch = 7 if c["COND_MODE"] == "onehot" else 3
    u = UNet2DConditionModel.from_pretrained(c["SD_MODEL_ID"], subfolder="unet",
                                             torch_dtype=torch.float16).to(device)
    u.requires_grad_(False)
    cn = ControlNetModel.from_pretrained(c["CONTROLNET_ID"], torch_dtype=torch.float32).to(device)
    if c["COND_MODE"] == "onehot":
        o = cn.controlnet_cond_embedding.conv_in
        cn.controlnet_cond_embedding.conv_in = torch.nn.Conv2d(
            cond_ch, o.out_channels, kernel_size=o.kernel_size, stride=o.stride,
            padding=o.padding).to(device)
        cn.register_to_config(conditioning_channels=cond_ch)
    if ck.get("lora_state"):
        from peft import LoraConfig
        u.add_adapter(LoraConfig(r=c["LORA_RANK"], lora_alpha=c["LORA_ALPHA"],
                                 lora_dropout=0.0, init_lora_weights="gaussian",
                                 target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
        u.load_state_dict({k: v.float() for k, v in ck["lora_state"].items()}, strict=False)
        for p in u.parameters():
            if p.requires_grad: p.data = p.data.float()
    if ck.get("control_state"):
        cn.load_state_dict({k: v.float() for k, v in ck["control_state"].items()})
    v = AutoencoderKL.from_pretrained(c["SD_MODEL_ID"], subfolder="vae").to(device).eval()
    te = CLIPTextModel.from_pretrained(c["SD_MODEL_ID"], subfolder="text_encoder",
                                       torch_dtype=torch.float16).to(device).eval()
    tk = CLIPTokenizer.from_pretrained(c["SD_MODEL_ID"], subfolder="tokenizer")
    return dict(unet=u.eval(), controlnet=cn.eval(), vae=v, text_encoder=te, tokenizer=tk,
                cfg=c, ck=ck, sched_cfg=DDIMScheduler.from_pretrained(
                    c["SD_MODEL_ID"], subfolder="scheduler").config)

@torch.no_grad()
def generate_from_hf(bundle, masks, steps=30, guidance=7.5, seed=0, batch=4, device="cuda"):
    import numpy as np, torch
    from diffusers import DDIMScheduler
    c = bundle["cfg"]; u, cn, v = bundle["unet"], bundle["controlnet"], bundle["vae"]
    cond_ch = 7 if c["COND_MODE"] == "onehot" else 3
    eye = np.eye(7, dtype=np.float32)
    def mk_cond(lab):
        code = np.where(lab == 255, 6, lab).astype(np.uint8)
        return eye[code].transpose(2, 0, 1)      # this helper covers the one-hot contract only
    def embed(caps):
        tok = bundle["tokenizer"](caps, padding="max_length", max_length=77, truncation=True,
                                  return_tensors="pt")
        return bundle["text_encoder"](tok.input_ids.to(device))[0]
    out = []
    for s in range(0, len(masks), batch):
        chunk = list(masks[s:s + batch]); b = len(chunk)
        cond = torch.from_numpy(np.stack([mk_cond(m) for m in chunk])).to(device)
        emb = embed(["aerial satellite view of Kathmandu Valley"] * b)
        neg = embed([""] * b)
        sch = DDIMScheduler.from_config(bundle["sched_cfg"]); sch.set_timesteps(steps, device=device)
        gg = torch.Generator(); gg.manual_seed(seed + s)
        lat = torch.randn(b, 4, 64, 64, generator=gg).to(device) * sch.init_noise_sigma
        for t in sch.timesteps:
            x = sch.scale_model_input(torch.cat([lat] * 2), t)
            with torch.cuda.amp.autocast():
                d, m = cn(x, t, encoder_hidden_states=torch.cat([neg, emb]),
                          controlnet_cond=torch.cat([cond, cond]), return_dict=False)
                e = u(x, t, encoder_hidden_states=torch.cat([neg, emb]),
                      down_block_additional_residuals=[a.to(u.dtype) for a in d],
                      mid_block_additional_residual=m.to(u.dtype)).sample
            e = e.float(); e_u, e_c = e.chunk(2); e = e_u + guidance * (e_c - e_u)
            lat = sch.step(e, t, lat).prev_sample
        img = v.decode((lat / v.config.scaling_factor).float()).sample
        img = ((img.float() / 2 + 0.5).clamp(0, 1).cpu().numpy().transpose(0, 2, 3, 1) * 255)
        out.append(img.round().astype(np.uint8))
    return np.concatenate(out)

# --- example ---
GEN_REPO = HF_REPO
if TRAINABLE:
    bundle = load_generator_from_hf(GEN_REPO, token=HF_TOKEN)
    demo = generate_from_hf(bundle, EVAL_LAB[:2], steps=cfg.DDIM_STEPS, seed=99)
    print(f"loaded {GEN_REPO} (step {bundle['ck']['step']}) | demo {demo.shape} {demo.dtype}")
    del bundle; gc.collect(); torch.cuda.empty_cache()
else:
    print("no trained checkpoint in this repo for the untrained ablations -- skipping the example")

## 23. Qualitative grids — real image / mask / generated

The counterpart of the segmentation notebooks' triptychs, and the figure a reader will look at
before they look at any number. Same palette, ignore renders black, so these sit next to the part-1
figures without a second legend.

What to look for, in order:
1. **Do the boundaries line up?** Not "is it pretty" — does the forest edge in the generated image
   fall where the mask says it does. That is the claim being made, and it is the one the eye is
   actually good at checking.
2. **Do rooftops look like Kathmandu rooftops?** Dense small footprints with metal and concrete
   colouring, not American suburbs or European tile. That is the LoRA's job and it fails visibly.
3. **Did the river survive?** At 0.57% of pixels it is a thread. A generator that paints a plausible
   valley and quietly omits the river will still look good in the first two checks — this is why the
   second figure isolates Road and River as binary overlays.

The third row of the second figure is the **re-segmentation**: the frozen SegFormer-B0's prediction
on the generated image, which is what produced the mIoU in section 19. Putting it beside the
conditioning mask makes the number legible — you can see whether the loss is boundary slop or a
whole class in the wrong place.


In [ ]:
REAL_RGB = [cv2.cvtColor(cv2.imread(str(CACHE / "images" / va_recs[i][2]), cv2.IMREAD_COLOR),
                         cv2.COLOR_BGR2RGB) for i in EVAL_IDX[:8]]

fig = triptych(REAL_RGB, EVAL_LAB[:8], GEN[:8], n=6,
               title=f"{RUN_NAME} / ablation {cfg.ABLATION} - step {final['best_step']}")
p1 = f"{cfg.OUT_DIR}/qual_triptych_{tag}.png"
plt.savefig(p1, dpi=130, bbox_inches="tight"); plt.show()

_pred = segment_rgb01(segmodel,
                      torch.from_numpy(GEN[:8].transpose(0, 3, 1, 2).astype(np.float32) / 255.0)).numpy()
fig, axes = plt.subplots(3, 6, figsize=(15, 7.6))
for j in range(6):
    axes[0, j].imshow(GEN[j]); axes[0, j].set_title(f"generated {j}", fontsize=8)
    axes[1, j].imshow(colorize(EVAL_LAB[j])); axes[1, j].set_title("conditioning mask", fontsize=8)
    axes[2, j].imshow(colorize(_pred[j])); axes[2, j].set_title("re-segmented", fontsize=8)
    for r in range(3): axes[r, j].axis("off")
fig.suptitle("What the frozen SegFormer-B0 sees in the generated image "
             f"(layout mIoU {lay['miou_gen']:.3f}, ceiling {lay['miou_real']:.3f})", fontsize=11)
fig.legend(handles=LEGEND, loc="lower center", ncol=4, fontsize=8, frameon=False)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
p2 = f"{cfg.OUT_DIR}/qual_resegment_{tag}.png"
plt.savefig(p2, dpi=130, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(2, 6, figsize=(15, 5.2))
for j in range(6):
    axes[0, j].imshow((EVAL_LAB[j] == 1) * 1.0 + (EVAL_LAB[j] == 2) * 0.5, cmap="gray")
    axes[0, j].set_title(f"mask {j}", fontsize=8); axes[0, j].axis("off")
    axes[1, j].imshow((_pred[j] == 1) * 1.0 + (_pred[j] == 2) * 0.5, cmap="gray")
    axes[1, j].set_title("re-segmented", fontsize=8); axes[1, j].axis("off")
fig.suptitle(f"Road (white) and River (grey) only - clDice Road "
             f"{RES['clDice']['Road_gen']:.3f}, River {RES['clDice']['River_gen']:.3f}", fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.94])
p3 = f"{cfg.OUT_DIR}/qual_thin_{tag}.png"
plt.savefig(p3, dpi=130, bbox_inches="tight"); plt.show()

for p in [p1, p2, p3]: hf_push(p, os.path.basename(p), "qualitative")
if WANDB: wandb.log({"qual/triptych": wandb.Image(p1), "qual/resegment": wandb.Image(p2)})

## 24. Ablation table

The paper's table, assembled across sessions. Each row is one configuration of section 2's ladder,
scored with the identical evaluation from section 19 on the identical `EVAL_IDX` masks.

**What runs where.** Rows **A** and **B** train nothing, so they can be evaluated inside any session
by swapping in a fresh off-the-shelf control branch and disabling the LoRA adapter — set
`RUN_UNTRAINED_ABLATIONS=True` and they cost about `2 x ABLATION_EVAL_N x gen_s` of generation time,
nothing else. Row **E** re-samples the current weights with the canny branch off, so it is nearly
free whenever this notebook is running D. Row **C** is the only one that needs its own training run:
set `cfg.ABLATION="C"` in a fresh session and run the notebook top to bottom.

Rows accumulate in `ablation_rows.csv` in the **D repo**, which acts as the table's home: every
session downloads it, adds or replaces its own rows by name, and pushes it back. That is what lets a
row computed in a session three days ago sit in the same table as one computed just now without
anybody copying numbers by hand.

Read the table in one direction: **A -> B** is what the palette remap alone buys, **B -> C** is what
domain adaptation buys, **C -> D** is what actually training the control branch on a one-hot
condition buys, and **D -> E** is what the canny branch is worth. If A and B do not sit near the
shuffled-null mIoU, the claim that the old pipeline ignored the mask is weaker than stated and the
paper should say so.


In [ ]:
import contextlib

@contextlib.contextmanager
def off_the_shelf(cond_mode):
    # temporarily swap in a fresh pretrained control branch and disable LoRA, for rows A and B
    global controlnet
    saved_cn, saved_mode = controlnet, cfg.COND_MODE
    lora_off = bool(cfg.USE_LORA and hasattr(unet, "disable_adapters"))
    fresh = ControlNetModel.from_pretrained(cfg.CONTROLNET_ID, torch_dtype=torch.float16).cuda().eval()
    fresh.requires_grad_(False)
    controlnet, cfg.COND_MODE = fresh, cond_mode
    if lora_off: unet.disable_adapters()
    try:
        yield
    finally:
        controlnet, cfg.COND_MODE = saved_cn, saved_mode
        if lora_off: unet.enable_adapters()
        del fresh; gc.collect(); torch.cuda.empty_cache()

def row_from(res):
    l = res["layout"]
    return {"row": res["name"], "cond": res.get("cond", ""), "lora": res.get("lora", ""),
            "control": res.get("control", ""), "canny": res.get("canny", ""),
            "miou_gen": round(l["miou_gen"], 4), "miou_shuffled": round(l["miou_shuffled"], 4),
            "miou_real": round(l["miou_real"], 4), "layout_ratio": round(l["layout_ratio"], 3),
            "clDice_Road": round(res["clDice"]["Road_gen"], 4),
            "clDice_River": round(res["clDice"]["River_gen"], 4),
            "KID_mean": round(res["KID_mean"], 5), "KID_std": round(res["KID_std"], 5),
            "FID": round(res["FID"], 2), "N_real": res["N_real"], "N_gen": res["N_gen"],
            "verdict_ratio": res["verdict_ratio"], "verdict_null": res["verdict_null"]}

ROWS = []
RES["cond"], RES["lora"] = cfg.COND_MODE, cfg.USE_LORA
RES["control"], RES["canny"] = ("trained" if cfg.TRAIN_CONTROLNET else "frozen"), cfg.USE_CANNY
ROWS.append(row_from(RES))

# ---- E: the same weights, sampled without the canny branch (free when running D) ----
if cfg.ABLATION == "D" and cfg.USE_CANNY:
    r_e, _ = evaluate_generated(EVAL_LAB[:cfg.ABLATION_EVAL_N], use_canny=False, seed=202, name="E")
    r_e.update(cond=cfg.COND_MODE, lora=cfg.USE_LORA, control="trained", canny=False)
    ROWS.append(row_from(r_e))
    print(f"E (no canny): mIoU {r_e['layout']['miou_gen']:.4f} | KID {r_e['KID_mean']:.5f}")

# ---- A and B: no training at all, so they can be evaluated here ----
if cfg.RUN_UNTRAINED_ABLATIONS:
    for letter, mode in (("A", "palette"), ("B", "ade20k")):
        with off_the_shelf(mode):
            r, _ = evaluate_generated(EVAL_LAB[:cfg.ABLATION_EVAL_N], seed=303, name=letter)
        r.update(cond=mode, lora=False, control="off-the-shelf", canny=cfg.USE_CANNY)
        ROWS.append(row_from(r))
        print(f"{letter} ({mode}, untrained): mIoU {r['layout']['miou_gen']:.4f} | "
              f"KID {r['KID_mean']:.5f}")

# ---- merge with whatever the table repo already holds ----
prev = hf_try_download("ablation_rows.csv", repo=ABLATION_TABLE_REPO)
TABLE = pd.read_csv(prev) if prev else pd.DataFrame()
new = pd.DataFrame(ROWS)
if len(TABLE):
    TABLE = TABLE[~TABLE.row.isin(new.row)]
TABLE = pd.concat([TABLE, new], ignore_index=True).sort_values("row")
tp = f"{cfg.OUT_DIR}/ablation_rows.csv"; TABLE.to_csv(tp, index=False)
hf_push(tp, "ablation_rows.csv", f"ablation rows from {tag}", repo=ABLATION_TABLE_REPO)

print("\n--- ABLATION TABLE (rows present so far) ---")
print(TABLE.to_string(index=False))
missing = sorted(set("ABCDE") - set(TABLE.row.astype(str)))
if missing:
    print(f"\nstill missing: {missing}  "
          "(C needs its own training run: set cfg.ABLATION='C' in a fresh session)")

if len(TABLE) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
    x = np.arange(len(TABLE))
    axes[0].bar(x, TABLE.miou_gen); axes[0].plot(x, TABLE.miou_shuffled, "k--", label="shuffled null")
    axes[0].plot(x, TABLE.miou_real, "r:", label="real-image ceiling")
    axes[0].set_xticks(x); axes[0].set_xticklabels(TABLE.row); axes[0].set_ylabel("layout mIoU")
    axes[0].legend(fontsize=7); axes[0].set_title("Layout fidelity")
    axes[1].bar(x, TABLE.KID_mean, yerr=TABLE.KID_std)
    axes[1].set_xticks(x); axes[1].set_xticklabels(TABLE.row); axes[1].set_ylabel("KID")
    axes[1].set_title(f"KID (N_real={RES['N_real']}, subsets of {RES['KID_subset_size']})")
    w = 0.38
    axes[2].bar(x - w/2, TABLE.clDice_Road, w, label="Road")
    axes[2].bar(x + w/2, TABLE.clDice_River, w, label="River")
    axes[2].set_xticks(x); axes[2].set_xticklabels(TABLE.row); axes[2].set_ylabel("clDice")
    axes[2].legend(fontsize=8); axes[2].set_title("Thin-structure survival")
    plt.tight_layout()
    pa = f"{cfg.OUT_DIR}/fig_ablation_{tag}.png"
    plt.savefig(pa, dpi=140, bbox_inches="tight"); plt.show()
    hf_push(pa, os.path.basename(pa), "ablation figure")

print("\n--- STAGE 2 SUMMARY ---")
print(json.dumps({"ablation": cfg.ABLATION, "best_step": final["best_step"],
                  "miou_gen": lay["miou_gen"], "miou_shuffled": lay["miou_shuffled"],
                  "miou_real": lay["miou_real"], "layout_ratio": lay["layout_ratio"],
                  "KID": RES["KID_mean"], "FID": RES["FID"],
                  "clDice_Road": RES["clDice"]["Road_gen"],
                  "clDice_River": RES["clDice"]["River_gen"],
                  "N_real": RES["N_real"], "N_gen": RES["N_gen"]}, indent=2, default=str))
print(f"hand-off: {cfg.SAMPLE_OUT_DIR}/genimage*.png + genmask*.png -> stage 4")
if WANDB: wandb.finish()

## 25. Run checklist

1. Internet **ON**; Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`; set `cfg.HF_USER`.
2. Section 5 must print **channel-order check PASSED** and an unmapped share of `0.000000%`. If it
   does not, stop — the condition is built from those pixels.
3. Section 6 repacks to the 512 cache (~15-20 min for 1224 tiles at 2345x2351), or set
   `PREBUILT_CACHE` to the cache the segmentation notebooks already built — it is byte-identical.
4. Section 8 must print **270 train tiles / 136 val tiles / 2 buffered out**, and the `train_sha1`
   must equal the one Stage 1 prints. Paste Stage 1's value into `cfg.STAGE1_TRAIN_SHA1` and it
   becomes an assert rather than a habit.
5. Section 12 must print `conditioning conv replaced: 3 -> 7 channels` and two **backward guard OK**
   lines. If the guard raises, gradient checkpointing has detached the graph — do not train past it.
6. **Section 14 is the go/no-go.** Read the measured peak VRAM and the projected hours before you
   walk away. If the projection exceeds two sessions, lower `MAX_STEPS` *now*, not later: the cosine
   schedule is defined over `MAX_STEPS`, so changing it mid-run makes the two halves incomparable.
7. Run top to bottom. `MAX_HOURS=10.5` stops the loop gracefully; re-running the notebook resumes
   from `last.pt`.
8. After section 19, read the three layout numbers in this order: **ratio**, then **lift over the
   shuffled null**, then the absolute mIoU.

### Reading the layout number honestly
SegFormer-B0's published fold-0 mIoU on **real** validation images is **0.2877**. That is the
ceiling for this measurement, so the specified absolute floor of 0.30 is above what a perfect
generator could achieve on this fold. Report `layout_ratio` as the primary number and the absolute
mIoU alongside it, and say plainly in the paper why. If you want an absolute number that clears 0.3,
evaluate with **TC-SegFormer** instead (fold-0 mIoU 0.3403) — but then say which segmenter produced
the number, because the choice of judge is now part of the claim. Swapping judges is a one-line
change to `cfg.SEG_REPO`; doing it *after* seeing the B0 result is not.

### If layout fidelity comes back FAIL
In order, cheapest and least protocol-invasive first:

- **Check the null first.** If `miou_gen` is close to `miou_shuffled`, the mask is being ignored
  entirely and no amount of extra training will fix a conditioning bug. Confirm section 12 printed
  the conv replacement, and confirm `COND_MODE` is `onehot`.
- **Train longer.** The zero-initialized conv means the branch starts by ignoring the condition and
  has to learn to use it; 3000 steps is modest. Raise `MAX_STEPS` (the cosine horizon moves with
  it) and resume.
- **Try `COND_CONV_INIT="palette_project"`.** It starts the branch already responding to the
  condition rather than having to discover it. This changes `ARCH_SIG`, so a resumed session will
  refuse the old checkpoint — that is intentional; start a fresh run and keep the old repo.
- **Raise `COND_SCALE` to 1.2-1.5 at sampling time.** Costs nothing, testable in minutes, and it is
  the one knob that separates "the branch learned nothing" from "the branch is being outvoted by the
  text prior at guidance 7.5". Lowering `GUIDANCE` to 5 tests the same thing from the other side.
- **Only then** consider raising `LORA_RANK` or `LR_CONTROL`.

Do **not** respond to a FAIL by lowering the thresholds, and do not re-pick the evaluation masks. The
thresholds are fixed here, before the numbers are seen, and should be quoted as-is.

### If you hit CUDA out of memory
The peak is in the control branch's backward, so it appears within the first few steps and is
predictable. In order:

1. Confirm section 12 printed `checkpointing True` and that section 14's measured peak is what you
   expected. That is the single biggest lever.
2. Halve the micro-batch and double the accumulation: `BATCH_SIZE=1`, `GRAD_ACCUM=16`. **Keep the
   product at 16** — that is the protocol; the split between the two is not.
3. If `bitsandbytes` failed to import, section 12 said so and you are paying ~2.2 GB extra for
   fp32 Adam moments. Fixing the wheel is worth more than any other single change.
4. If **sampling** OOMs rather than training, lower `EVAL_BATCH`. It has no effect on any result.

### Chaining with Stage 1
Section 21 consumes class-index PNGs and does not care where they came from. Point `SAMPLE_IN_DIR`
at Stage 1's `gen_masks_512` directory and the same cell produces fake pairs from fake masks. Until
Stage 1 lands, leaving it `None` produces fake images on **real** held-out masks, which is a
perfectly good stage-4 augmentation set on its own and is the safer thing to have ready first.
